## Hours of very high thermal stress total

In [ ]:
import os
import gc
import time
import numpy as np
import xarray as xr
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from shapely.geometry import shape, Point
from shapely.ops import unary_union
from shapely.prepared import prep
from cartopy.io import shapereader

# ── Config ────────────────────────────────────────────────────────────────────
BASE_PATTERN = r"C:\Users\nerc-user\OneDrive - Nexus365\UTCI_NN\ERA5_exploration\data\thermal_stress_hrs\utci_hours_diff_2024-{month:02d}.nc"
VAR_NAME     = "hours_diff_nn_minus_poly"
CATEGORIES   = ["strong_heat", "extreme_heat", "very_strong_heat",
                "cold", "extreme_cold", "very_cold"]
MONTHS       = list(range(1, 13))
RESOLUTION   = "110m"

MASK_CACHE   = "land_mask_110m.nc"
OUTPUT_NC    = "utci_yearly_land_hours.nc"

INPUT_NC     = "utci_yearly_land_hours.nc"
OUTPUT_PNG   = "utci_four_categories_2x2.png"
OUTPUT_PNG_AB = "utci_ab_categories_1x2.png"
COASTLINE_R  = "110m"
N_BINS       = 13
MAP_EXTENT   = [-180, 180, -60, 90]
CBAR_LABEL   = "NN − Polynomial UTCI-Stress Difference (hours)"

PANEL_CONFIG = [
    dict(var="strong_heat",      title="Heat Stress (≥ 32 °C)",              cmap="RdBu_r"),
    dict(var="extreme_heat",     title="Extreme Heat Stress (> 46 °C)",       cmap="RdBu_r"),
    dict(var="cold",             title="Cold Stress (< -13 °C)",              cmap="RdBu_r"),
    dict(var="extreme_cold",     title="Extreme Cold Stress (< -40 °C)",      cmap="RdBu_r"),
]

PANEL_CONFIG_AB = [
    dict(var="very_cold",        title="Very Strong Cold Stress (< -27 °C)",  cmap="RdBu_r"),
    dict(var="very_strong_heat", title="Very Strong Heat Stress (> 38 °C)",   cmap="RdBu_r"),
]

# PANEL_CONFIG_AB = [
#     dict(var="extreme_cold",     title="Extreme Cold Stress (< -40 °C)",      cmap="RdBu_r"),
#     dict(var="extreme_heat",     title="Extreme Heat Stress (> 46 °C)",       cmap="RdBu_r"),
# ]


# ── Helpers ───────────────────────────────────────────────────────────────────
def _build_file_list():
    files = []
    for m in MONTHS:
        p = BASE_PATTERN.format(month=m)
        if not os.path.exists(p):
            raise FileNotFoundError(p)
        files.append(p)
    return files


def _load_concat(files):
    ds_list = []
    for m, f in enumerate(files, start=1):
        ds = xr.open_dataset(f).expand_dims(month=[m])
        ds_list.append(ds)
    return xr.concat(ds_list, dim="month")


def _sum_to_hours(ds, cat):
    """Sum hours across all months for one category."""
    return (
        ds.sel(category=cat)[VAR_NAME]
          .sum(dim="month")
          .squeeze(drop=True)
          .drop_vars("category", errors="ignore")
    )


def _build_or_load_mask(lon, lat):
    """Land mask: True = land. Cached to MASK_CACHE."""
    if os.path.exists(MASK_CACHE):
        cached = xr.open_dataarray(MASK_CACHE)
        if np.array_equal(cached["lon"].values, lon) and \
           np.array_equal(cached["lat"].values, lat):
            print(f"  Loaded land mask from {MASK_CACHE}")
            return cached
        cached.close()
        os.remove(MASK_CACHE)
        print("  Mask coords changed – rebuilding.")

    print("  Building land mask …")
    shp   = shapereader.natural_earth(RESOLUTION, "physical", "land")
    geoms = [shape(r.geometry) for r in shapereader.Reader(shp).records()]
    land  = prep(unary_union(geoms))

    lon2d, lat2d = np.meshgrid(lon, lat)
    flat  = lon2d.size
    mask  = np.empty(flat, dtype=bool)
    chunk = 2_000_000
    t0    = time.time()
    for i in range(0, flat, chunk):
        pts = [Point(xy) for xy in zip(lon2d.ravel()[i:i+chunk],
                                       lat2d.ravel()[i:i+chunk])]
        mask[i:i+chunk] = [land.contains(p) for p in pts]
    print(f"  Done in {time.time()-t0:.1f}s")

    da = xr.DataArray(mask.reshape(lat2d.shape),
                      coords={"lat": lat, "lon": lon},
                      dims=("lat", "lon"))
    da.to_netcdf(MASK_CACHE)
    print(f"  Saved mask → {MASK_CACHE}")
    return da


def _discrete_cmap(vabs, cmap_name, n):
    """Symmetric diverging colormap centred on 0."""
    boundaries = np.linspace(-vabs, vabs, n + 1)
    cmap = plt.get_cmap(cmap_name, n)
    norm = mpl.colors.BoundaryNorm(boundaries, ncolors=n)
    return cmap, boundaries, norm


def _map_features(ax):
    ax.add_feature(cfeature.COASTLINE.with_scale(COASTLINE_R), linewidth=0.4)
    ax.add_feature(cfeature.BORDERS.with_scale(COASTLINE_R),   linewidth=0.2)
    ax.add_feature(cfeature.OCEAN, facecolor="lightcyan",       zorder=0)
    ax.add_feature(cfeature.LAND,  facecolor="whitesmoke",      zorder=0)


# ── Preprocess ────────────────────────────────────────────────────────────────
def preprocess():
    if os.path.exists(OUTPUT_NC):
        gc.collect()
        os.remove(OUTPUT_NC)
        print(f"Deleted existing {OUTPUT_NC}, reprocessing …")

    print("Loading monthly files …")
    ds_all = _load_concat(_build_file_list())
    lon    = ds_all["lon"].values
    lat    = ds_all["lat"].values

    print("Building land mask …")
    mask = _build_or_load_mask(lon, lat)

    print("Summing categories …")
    data_vars = {}
    for cat in CATEGORIES:
        arr = _sum_to_hours(ds_all, cat)
        arr = arr.where(mask, other=np.nan)
        data_vars[cat] = arr
        print(f"  {cat}: {float(arr.min()):.1f} – {float(arr.max()):.1f} hours")

    ds_out = xr.Dataset(data_vars)
    ds_out.attrs["description"] = "UTCI yearly totals (hours), land-only, 2024"
    ds_out.to_netcdf(OUTPUT_NC)
    print(f"\nSaved → {OUTPUT_NC}")



# ── Plot (1x2, panels a & b) ──────────────────────────────────────────────────
def _map_features_white_ocean(ax):
    ax.add_feature(cfeature.COASTLINE.with_scale(COASTLINE_R), linewidth=0.4)
    ax.add_feature(cfeature.BORDERS.with_scale(COASTLINE_R),   linewidth=0.2)
    ax.add_feature(cfeature.OCEAN, facecolor="white",           zorder=0)
    ax.add_feature(cfeature.LAND,  facecolor="whitesmoke",      zorder=0)
    
def plot_ab(savepath=OUTPUT_PNG_AB):
    ds           = xr.open_dataset(INPUT_NC)
    lon          = ds["lon"].values
    lat          = ds["lat"].values
    lon2d, lat2d = np.meshgrid(lon, lat)
    proj         = ccrs.PlateCarree()

    fig = plt.figure(figsize=(11, 4.5))
    gs  = fig.add_gridspec(
        2, 2,
        height_ratios=[1, 0.04],
        hspace=0.08, wspace=0.12,
        top=0.92, bottom=0.12,
    )

    axes     = [fig.add_subplot(gs[0, c], projection=proj) for c in range(2)]
    cbar_axs = [fig.add_subplot(gs[1, c])                  for c in range(2)]

    for ax in axes:
        ax.set_aspect(1.2)

    for idx, (ax, cax, cfg) in enumerate(zip(axes, cbar_axs, PANEL_CONFIG_AB)):
        data = ds[cfg["var"]].values
        vabs = float(np.nanpercentile(np.abs(data), 99))
        if not np.isfinite(vabs) or vabs == 0:
            vabs = 1.0

        cmap, boundaries, norm = _discrete_cmap(vabs, cfg["cmap"], N_BINS)

        _map_features_white_ocean(ax)
        ax.set_extent(MAP_EXTENT, crs=proj)
        ax.set_title(cfg["title"], fontsize=11, pad=4)
        ax.text(-0.07, 1.05, "ab"[idx],
                transform=ax.transAxes,
                fontsize=13, fontweight="bold", va="top", ha="left")

        ax.pcolormesh(lon2d, lat2d, data,
                      transform=proj, cmap=cmap, norm=norm,
                      shading="auto", zorder=1)

        cb = fig.colorbar(
            mpl.cm.ScalarMappable(norm=norm, cmap=cmap),
            cax=cax, orientation="horizontal",
            boundaries=boundaries, ticks=boundaries,
            extend="both",
        )
        cb.set_label(CBAR_LABEL, fontsize=8)
        cb.ax.tick_params(labelsize=7, rotation=35)
        cb.ax.xaxis.set_major_formatter(mpl.ticker.FormatStrFormatter("%.0f"))

    fig.suptitle("2024 Total Difference in UTCI-Stress Hours",
                 fontsize=14, y=1.005)

    if savepath:
        fig.savefig(savepath, dpi=plt.rcParams["savefig.dpi"], bbox_inches="tight")
        print(f"Saved → {savepath}")

    plt.show()
    ds.close()

# ── Entry point ───────────────────────────────────────────────────────────────
preprocess()
plot_ab()

In [ ]:
ds = xr.open_dataset("utci_yearly_land_hours.nc")

# Inspect variables (pick the right one)
print(ds)

# Replace 'variable_name' with the actual variable in your dataset
var = ds['extreme_heat']

# If there's a time dimension, reduce it (e.g., mean or max over time)
# Uncomment if needed:
# var = var.mean(dim='time')

# Find max and min values
max_val = var.max()
min_val = var.min()

print("Max value:", float(max_val.values))
print("Min value:", float(min_val.values))

# Get indices of max and min
max_idx = np.unravel_index(np.argmax(var.values), var.shape)
min_idx = np.unravel_index(np.argmin(var.values), var.shape)

print("Max index:", max_idx)
print("Min index:", min_idx)

# If coordinates exist (lat/lon), extract them
coords = list(var.dims)

max_coords = {dim: var[dim].values[idx] for dim, idx in zip(coords, max_idx)}
min_coords = {dim: var[dim].values[idx] for dim, idx in zip(coords, min_idx)}

print("Max location (coords):", max_coords)
print("Min location (coords):", min_coords)

In [ ]:
543/24

In [ ]:
import os
import gc
import time
import numpy as np
import xarray as xr
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from shapely.geometry import shape, Point
from shapely.ops import unary_union
from shapely.prepared import prep
from cartopy.io import shapereader

# ── Config ────────────────────────────────────────────────────────────────────
BASE_PATTERN = r"C:\Users\nerc-user\OneDrive - Nexus365\UTCI_NN\ERA5_exploration\data\thermal_stress_hrs\utci_hours_diff_2024-{month:02d}.nc"
VAR_DIFF     = "hours_diff_nn_minus_poly"
VAR_BASE     = "hours_utci_polynomial"
CATEGORIES   = ["strong_heat", "extreme_heat", "very_strong_heat",
                "cold", "extreme_cold", "very_cold"]
MONTHS       = list(range(1, 13))
RESOLUTION   = "110m"

MASK_CACHE   = "land_mask_110m.nc"
OUTPUT_NC    = "utci_yearly_land_pct_change.nc"

INPUT_NC     = "utci_yearly_land_pct_change.nc"
OUTPUT_PNG   = "utci_four_categories_2x2_pct.png"
OUTPUT_PNG_AB = "utci_ab_categories_1x2_pct.png"
COASTLINE_R  = "110m"
N_BINS       = 13
MAP_EXTENT   = [-180, 180, -60, 90]
CBAR_LABEL   = "NN − Polynomial UTCI-Stress Difference (%)"

PANEL_CONFIG = [
    dict(var="strong_heat",      title="Heat Stress (≥ 32 °C)",              cmap="RdBu_r"),
    dict(var="extreme_heat",     title="Extreme Heat Stress (> 46 °C)",       cmap="RdBu_r"),
    dict(var="cold",             title="Cold Stress (< -13 °C)",              cmap="RdBu_r"),
    dict(var="extreme_cold",     title="Extreme Cold Stress (< -40 °C)",      cmap="RdBu_r"),
]

PANEL_CONFIG_AB = [
    dict(var="very_cold",        title="Very Strong Cold Stress (< -27 °C)",  cmap="RdBu_r"),
    dict(var="very_strong_heat", title="Very Strong Heat Stress (> 38 °C)",   cmap="RdBu_r"),
]

PANEL_CONFIG_AB = [
    dict(var="cold",             title="Cold Stress (< -13 °C)",              cmap="RdBu_r"),
    dict(var="strong_heat",      title="Heat Stress (≥ 32 °C)",              cmap="RdBu_r"),
]

PANEL_CONFIG_AB = [
    dict(var="extreme_cold",     title="Extreme Cold Stress (< -40 °C)",      cmap="RdBu_r"),
    dict(var="extreme_heat",     title="Extreme Heat Stress (> 46 °C)",       cmap="RdBu_r"),
]


# ── Helpers ───────────────────────────────────────────────────────────────────
def _build_file_list():
    files = []
    for m in MONTHS:
        p = BASE_PATTERN.format(month=m)
        if not os.path.exists(p):
            raise FileNotFoundError(p)
        files.append(p)
    return files


def _load_concat(files):
    ds_list = []
    for m, f in enumerate(files, start=1):
        ds = xr.open_dataset(f).expand_dims(month=[m])
        ds_list.append(ds)
    return xr.concat(ds_list, dim="month")


def _sum_var(ds, cat, var_name):
    """Sum a variable across all months for one category."""
    return (
        ds.sel(category=cat)[var_name]
          .sum(dim="month")
          .squeeze(drop=True)
          .drop_vars("category", errors="ignore")
    )


def _build_or_load_mask(lon, lat):
    """Land mask: True = land. Cached to MASK_CACHE."""
    if os.path.exists(MASK_CACHE):
        cached = xr.open_dataarray(MASK_CACHE)
        if np.array_equal(cached["lon"].values, lon) and \
           np.array_equal(cached["lat"].values, lat):
            print(f"  Loaded land mask from {MASK_CACHE}")
            return cached
        cached.close()
        os.remove(MASK_CACHE)
        print("  Mask coords changed – rebuilding.")

    print("  Building land mask …")
    shp   = shapereader.natural_earth(RESOLUTION, "physical", "land")
    geoms = [shape(r.geometry) for r in shapereader.Reader(shp).records()]
    land  = prep(unary_union(geoms))

    lon2d, lat2d = np.meshgrid(lon, lat)
    flat  = lon2d.size
    mask  = np.empty(flat, dtype=bool)
    chunk = 2_000_000
    t0    = time.time()
    for i in range(0, flat, chunk):
        pts = [Point(xy) for xy in zip(lon2d.ravel()[i:i+chunk],
                                       lat2d.ravel()[i:i+chunk])]
        mask[i:i+chunk] = [land.contains(p) for p in pts]
    print(f"  Done in {time.time()-t0:.1f}s")

    da = xr.DataArray(mask.reshape(lat2d.shape),
                      coords={"lat": lat, "lon": lon},
                      dims=("lat", "lon"))
    da.to_netcdf(MASK_CACHE)
    print(f"  Saved mask → {MASK_CACHE}")
    return da


def _discrete_cmap(vabs, cmap_name, n):
    """Symmetric diverging colormap centred on 0."""
    boundaries = np.linspace(-vabs, vabs, n + 1)
    cmap = plt.get_cmap(cmap_name, n)
    norm = mpl.colors.BoundaryNorm(boundaries, ncolors=n)
    return cmap, boundaries, norm


def _map_features(ax):
    ax.add_feature(cfeature.COASTLINE.with_scale(COASTLINE_R), linewidth=0.4)
    ax.add_feature(cfeature.BORDERS.with_scale(COASTLINE_R),   linewidth=0.2)
    ax.add_feature(cfeature.OCEAN, facecolor="lightcyan",       zorder=0)
    ax.add_feature(cfeature.LAND,  facecolor="whitesmoke",      zorder=0)


def _map_features_white_ocean(ax):
    ax.add_feature(cfeature.COASTLINE.with_scale(COASTLINE_R), linewidth=0.4)
    ax.add_feature(cfeature.BORDERS.with_scale(COASTLINE_R),   linewidth=0.2)
    ax.add_feature(cfeature.OCEAN, facecolor="white",           zorder=0)
    ax.add_feature(cfeature.LAND,  facecolor="whitesmoke",      zorder=0)


# ── Preprocess ────────────────────────────────────────────────────────────────
def preprocess():
    if os.path.exists(OUTPUT_NC):
        gc.collect()
        os.remove(OUTPUT_NC)
        print(f"Deleted existing {OUTPUT_NC}, reprocessing …")

    print("Loading monthly files …")
    ds_all = _load_concat(_build_file_list())
    lon    = ds_all["lon"].values
    lat    = ds_all["lat"].values

    print("Building land mask …")
    mask = _build_or_load_mask(lon, lat)

    print("Computing percentage change per category …")
    data_vars = {}
    for cat in CATEGORIES:
        diff = _sum_var(ds_all, cat, VAR_DIFF)   # NN − Poly (hours)
        base = _sum_var(ds_all, cat, VAR_BASE)   # Polynomial baseline (hours)

        # Percentage change; mask cells where baseline is ~0 to avoid div-by-zero
        pct = (diff / base.where(np.abs(base) > 0.5)) * 100

        # Apply land mask
        pct = pct.where(mask, other=np.nan)

        data_vars[cat] = pct
        print(f"  {cat}: {float(np.nanmin(pct.values)):.1f} – {float(np.nanmax(pct.values)):.1f} %")

    ds_out = xr.Dataset(data_vars)
    ds_out.attrs["description"] = "UTCI yearly % change (NN vs Polynomial), land-only, 2024"
    ds_out.to_netcdf(OUTPUT_NC)
    print(f"\nSaved → {OUTPUT_NC}")


# ── Plot (2x2, four categories) ───────────────────────────────────────────────
def plot_four(savepath=OUTPUT_PNG):
    ds           = xr.open_dataset(INPUT_NC)
    lon          = ds["lon"].values
    lat          = ds["lat"].values
    lon2d, lat2d = np.meshgrid(lon, lat)
    proj         = ccrs.PlateCarree()

    fig = plt.figure(figsize=(14, 8))
    gs  = fig.add_gridspec(
        3, 2,
        height_ratios=[1, 1, 0.04],
        hspace=0.15, wspace=0.12,
        top=0.92, bottom=0.08,
    )

    axes     = [fig.add_subplot(gs[r, c], projection=proj)
                for r in range(2) for c in range(2)]
    cbar_axs = [fig.add_subplot(gs[2, c]) for c in range(2)]

    labels = "abcd"
    for idx, (ax, cfg) in enumerate(zip(axes, PANEL_CONFIG)):
        data = ds[cfg["var"]].values
        vabs = float(np.nanpercentile(np.abs(data), 99))
        if not np.isfinite(vabs) or vabs == 0:
            vabs = 1.0

        cmap, boundaries, norm = _discrete_cmap(vabs, cfg["cmap"], N_BINS)

        _map_features_white_ocean(ax)
        ax.set_extent(MAP_EXTENT, crs=proj)
        ax.set_title(cfg["title"], fontsize=10, pad=4)
        ax.text(-0.07, 1.05, labels[idx],
                transform=ax.transAxes,
                fontsize=13, fontweight="bold", va="top", ha="left")

        ax.pcolormesh(lon2d, lat2d, data,
                      transform=proj, cmap=cmap, norm=norm,
                      shading="auto", zorder=1)

        # Shared colorbars per column
        if idx in (0, 1):
            col = idx % 2
            cb = fig.colorbar(
                mpl.cm.ScalarMappable(norm=norm, cmap=cmap),
                cax=cbar_axs[col], orientation="horizontal",
                boundaries=boundaries, ticks=boundaries,
                extend="both",
            )
            cb.set_label(CBAR_LABEL, fontsize=8)
            cb.ax.tick_params(labelsize=7, rotation=35)
            cb.ax.xaxis.set_major_formatter(mpl.ticker.FormatStrFormatter("%.0f"))

    fig.suptitle("2024 Percentage Change in UTCI-Stress Hours (NN vs Polynomial)",
                 fontsize=13, y=1.005)

    if savepath:
        fig.savefig(savepath, dpi=plt.rcParams["savefig.dpi"], bbox_inches="tight")
        print(f"Saved → {savepath}")

    plt.show()
    ds.close()


# ── Plot (1x2, panels a & b) ──────────────────────────────────────────────────
def plot_ab(savepath=OUTPUT_PNG_AB):
    ds           = xr.open_dataset(INPUT_NC)
    lon          = ds["lon"].values
    lat          = ds["lat"].values
    lon2d, lat2d = np.meshgrid(lon, lat)
    proj         = ccrs.PlateCarree()

    fig = plt.figure(figsize=(11, 4.5))
    gs  = fig.add_gridspec(
        2, 2,
        height_ratios=[1, 0.04],
        hspace=0.08, wspace=0.12,
        top=0.92, bottom=0.12,
    )

    axes     = [fig.add_subplot(gs[0, c], projection=proj) for c in range(2)]
    cbar_axs = [fig.add_subplot(gs[1, c])                  for c in range(2)]

    for ax in axes:
        ax.set_aspect(1.2)

    for idx, (ax, cax, cfg) in enumerate(zip(axes, cbar_axs, PANEL_CONFIG_AB)):
        data = ds[cfg["var"]].values
        vabs = float(np.nanpercentile(np.abs(data), 99))
        if not np.isfinite(vabs) or vabs == 0:
            vabs = 1.0

        cmap, boundaries, norm = _discrete_cmap(vabs, cfg["cmap"], N_BINS)

        _map_features_white_ocean(ax)
        ax.set_extent(MAP_EXTENT, crs=proj)
        ax.set_title(cfg["title"], fontsize=11, pad=4)
        ax.text(-0.07, 1.05, "ab"[idx],
                transform=ax.transAxes,
                fontsize=13, fontweight="bold", va="top", ha="left")

        ax.pcolormesh(lon2d, lat2d, data,
                      transform=proj, cmap=cmap, norm=norm,
                      shading="auto", zorder=1)

        cb = fig.colorbar(
            mpl.cm.ScalarMappable(norm=norm, cmap=cmap),
            cax=cax, orientation="horizontal",
            boundaries=boundaries, ticks=boundaries,
            extend="both",
        )
        cb.set_label(CBAR_LABEL, fontsize=8)
        cb.ax.tick_params(labelsize=7, rotation=35)
        cb.ax.xaxis.set_major_formatter(mpl.ticker.FormatStrFormatter("%.0f"))

    fig.suptitle("2024 Percentage Change in UTCI-Stress Hours (NN vs Polynomial)",
                 fontsize=14, y=1.005)

    if savepath:
        fig.savefig(savepath, dpi=plt.rcParams["savefig.dpi"], bbox_inches="tight")
        print(f"Saved → {savepath}")

    plt.show()
    ds.close()


# ── Entry point ───────────────────────────────────────────────────────────────
preprocess()
plot_ab()

In [ ]:
import os
import gc
import time
import numpy as np
import xarray as xr
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from shapely.geometry import shape, Point
from shapely.ops import unary_union
from shapely.prepared import prep
from cartopy.io import shapereader

# ── Style ────────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 300,
    'savefig.dpi': 600,
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'Times New Roman', 'Times New Roman'],
    'font.size': 12,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'axes.labelpad': 0.5,
    'axes.linewidth': 1.0,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 11,
    'legend.frameon': False,
    'lines.linewidth': 1.5,
    'lines.markersize': 6,
    'axes.grid': False,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
})

# ── Config ────────────────────────────────────────────────────────────────────
BASE_PATTERN = r"C:\Users\nerc-user\OneDrive - Nexus365\UTCI_NN\ERA5_exploration\data\thermal_stress_hrs\utci_hours_diff_2024-{month:02d}.nc"
VAR_DIFF     = "hours_diff_nn_minus_poly"
VAR_BASE     = "hours_utci_polynomial"
CATEGORIES   = ["strong_heat", "extreme_heat", "very_strong_heat",
                "cold", "extreme_cold", "very_cold"]
MONTHS       = list(range(1, 13))
RESOLUTION   = "110m"

MASK_CACHE    = "land_mask_110m.nc"
OUTPUT_NC     = "utci_yearly_land_pct_change.nc"
INPUT_NC      = "utci_yearly_land_pct_change.nc"

OUTPUT_PNG    = "utci_four_categories_2x2_pct.png"
OUTPUT_PNG_AB = "utci_ab_categories_1x2_pct.png"

COASTLINE_R = "110m"
N_BINS      = 13
MAP_EXTENT  = [-180, 180, -60, 90]
CBAR_LABEL  = "NN − Polynomial UTCI-Stress Difference (%)"

PANEL_CONFIG = [
    dict(var="strong_heat",   title="Heat Stress (≥ 32 °C)",         cmap="RdBu_r"),
    dict(var="extreme_heat",  title="Extreme Heat Stress (> 46 °C)",  cmap="RdBu_r"),
    dict(var="cold",          title="Cold Stress (< -13 °C)",        cmap="RdBu_r"),
    dict(var="extreme_cold",  title="Extreme Cold Stress (< -40 °C)", cmap="RdBu_r"),
]

# Keep only one AB definition
PANEL_CONFIG_AB = [
    dict(var="extreme_cold", title="Extreme Cold Stress (< -40 °C)", cmap="RdBu_r"),
    dict(var="extreme_heat", title="Extreme Heat Stress (> 46 °C)",  cmap="RdBu_r"),
]

# ── Helpers ───────────────────────────────────────────────────────────────────
def _build_file_list():
    files = []
    for m in MONTHS:
        p = BASE_PATTERN.format(month=m)
        if not os.path.exists(p):
            raise FileNotFoundError(p)
        files.append(p)
    return files


def _load_concat(files):
    ds_list = []
    for m, f in enumerate(files, start=1):
        ds = xr.open_dataset(f).expand_dims(month=[m])
        ds_list.append(ds)
    return xr.concat(ds_list, dim="month")


def _sum_var(ds, cat, var_name):
    return (
        ds.sel(category=cat)[var_name]
          .sum(dim="month")
          .squeeze(drop=True)
          .drop_vars("category", errors="ignore")
    )


def _build_or_load_mask(lon, lat):
    if os.path.exists(MASK_CACHE):
        cached = xr.open_dataarray(MASK_CACHE)
        if np.array_equal(cached["lon"].values, lon) and np.array_equal(cached["lat"].values, lat):
            print(f"  Loaded land mask from {MASK_CACHE}")
            return cached
        cached.close()
        os.remove(MASK_CACHE)
        print("  Mask coords changed – rebuilding.")

    print("  Building land mask …")
    shp   = shapereader.natural_earth(RESOLUTION, "physical", "land")
    geoms = [shape(r.geometry) for r in shapereader.Reader(shp).records()]
    land  = prep(unary_union(geoms))

    lon2d, lat2d = np.meshgrid(lon, lat)
    flat  = lon2d.size
    mask  = np.empty(flat, dtype=bool)
    chunk = 2_000_000

    t0 = time.time()
    for i in range(0, flat, chunk):
        pts = [Point(xy) for xy in zip(lon2d.ravel()[i:i+chunk],
                                       lat2d.ravel()[i:i+chunk])]
        mask[i:i+chunk] = [land.contains(p) for p in pts]
    print(f"  Done in {time.time()-t0:.1f}s")

    da = xr.DataArray(
        mask.reshape(lat2d.shape),
        coords={"lat": lat, "lon": lon},
        dims=("lat", "lon")
    )
    da.to_netcdf(MASK_CACHE)
    print(f"  Saved mask → {MASK_CACHE}")
    return da


def _discrete_cmap(vabs, cmap_name, n):
    boundaries = np.linspace(-vabs, vabs, n + 1)
    cmap = plt.get_cmap(cmap_name, n)
    norm = mpl.colors.BoundaryNorm(boundaries, ncolors=n)
    return cmap, boundaries, norm


def _map_features_white_ocean(ax):
    ax.add_feature(cfeature.COASTLINE.with_scale(COASTLINE_R), linewidth=0.4)
    ax.add_feature(cfeature.BORDERS.with_scale(COASTLINE_R), linewidth=0.2)
    ax.add_feature(cfeature.OCEAN, facecolor="white", zorder=0)
    ax.add_feature(cfeature.LAND, facecolor="whitesmoke", zorder=0)
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)


# ── Preprocess ────────────────────────────────────────────────────────────────
def preprocess():
    if os.path.exists(OUTPUT_NC):
        gc.collect()
        os.remove(OUTPUT_NC)
        print(f"Deleted existing {OUTPUT_NC}, reprocessing …")

    print("Loading monthly files …")
    ds_all = _load_concat(_build_file_list())
    lon = ds_all["lon"].values
    lat = ds_all["lat"].values

    print("Building land mask …")
    mask = _build_or_load_mask(lon, lat)

    print("Computing percentage change per category …")
    data_vars = {}
    for cat in CATEGORIES:
        diff = _sum_var(ds_all, cat, VAR_DIFF)
        base = _sum_var(ds_all, cat, VAR_BASE)

        pct = (diff / base.where(np.abs(base) > 0.5)) * 100
        pct = pct.where(mask, other=np.nan)

        data_vars[cat] = pct
        print(f"  {cat}: {float(np.nanmin(pct.values)):.1f} – {float(np.nanmax(pct.values)):.1f} %")

    ds_out = xr.Dataset(data_vars)
    ds_out.attrs["description"] = "UTCI yearly % change (NN vs Polynomial), land-only, 2024"
    ds_out.to_netcdf(OUTPUT_NC)
    print(f"\nSaved → {OUTPUT_NC}")


# ── Plot: 2x2 ────────────────────────────────────────────────────────────────
def plot_four(savepath=OUTPUT_PNG):
    ds = xr.open_dataset(INPUT_NC)
    lon = ds["lon"].values
    lat = ds["lat"].values
    lon2d, lat2d = np.meshgrid(lon, lat)

    proj = ccrs.PlateCarree()

    fig = plt.figure(figsize=(12, 8))
    gs = fig.add_gridspec(
        2, 2,
        left=0.05, right=0.88,
        bottom=0.08, top=0.95,
        hspace=0.04, wspace=0.06
    )

    axes = np.empty((2, 2), dtype=object)
    for r in range(2):
        for c in range(2):
            axes[r, c] = fig.add_subplot(gs[r, c], projection=proj)

    # shared side colourbars
    cax1 = fig.add_axes([0.90, 0.56, 0.010, 0.30])
    cax2 = fig.add_axes([0.90, 0.15, 0.010, 0.30])

    panel_labels = ["a", "b", "c", "d"]

    im_top = None
    im_bottom = None

    for idx, (ax, cfg) in enumerate(zip(axes.flat, PANEL_CONFIG)):
        data = ds[cfg["var"]].values
        vabs = float(np.nanpercentile(np.abs(data), 99))
        if not np.isfinite(vabs) or vabs == 0:
            vabs = 1.0

        cmap, boundaries, norm = _discrete_cmap(vabs, cfg["cmap"], N_BINS)

        _map_features_white_ocean(ax)
        ax.set_extent(MAP_EXTENT, crs=proj)
        ax.set_title(cfg["title"], fontsize=13, pad=4)
        ax.text(-0.08, 1.05, panel_labels[idx],
                transform=ax.transAxes,
                fontsize=14, fontweight="bold",
                va="top", ha="left")

        im = ax.pcolormesh(
            lon2d, lat2d, data,
            transform=proj, cmap=cmap, norm=norm,
            shading="auto", zorder=1
        )

        if idx < 2:
            im_top = im
        else:
            im_bottom = im

    cb1 = fig.colorbar(
        mpl.cm.ScalarMappable(norm=im_top.norm, cmap=im_top.cmap),
        cax=cax1, orientation="vertical",
        extend="both", extendfrac=0.06
    )
    cb1.set_label(CBAR_LABEL, fontsize=12, labelpad=12)
    cb1.ax.tick_params(labelsize=10, pad=4)

    cb2 = fig.colorbar(
        mpl.cm.ScalarMappable(norm=im_bottom.norm, cmap=im_bottom.cmap),
        cax=cax2, orientation="vertical",
        extend="both", extendfrac=0.06
    )
    cb2.set_label(CBAR_LABEL, fontsize=12, labelpad=12)
    cb2.ax.tick_params(labelsize=10, pad=4)

    fig.savefig(savepath, dpi=plt.rcParams["savefig.dpi"], bbox_inches="tight", pad_inches=0.01)
    print(f"Saved → {savepath}")
    plt.show()
    ds.close()


# ── Plot: 1x2 ────────────────────────────────────────────────────────────────
def plot_ab(savepath=OUTPUT_PNG_AB):
    ds = xr.open_dataset(INPUT_NC)
    lon = ds["lon"].values
    lat = ds["lat"].values
    lon2d, lat2d = np.meshgrid(lon, lat)

    proj = ccrs.PlateCarree()

    fig = plt.figure(figsize=(12, 4.8))
    gs = fig.add_gridspec(
        1, 2,
        left=0.05, right=0.88,
        bottom=0.12, top=0.90,
        wspace=0.06
    )

    axes = [fig.add_subplot(gs[0, c], projection=proj) for c in range(2)]

    cax1 = fig.add_axes([0.90, 0.20, 0.010, 0.55])
    cax2 = fig.add_axes([0.90, 0.20, 0.010, 0.55])  # same side, adjust if you want separate bars

    for ax, cfg, label in zip(axes, PANEL_CONFIG_AB, ["a", "b"]):
        data = ds[cfg["var"]].values
        vabs = float(np.nanpercentile(np.abs(data), 99))
        if not np.isfinite(vabs) or vabs == 0:
            vabs = 1.0

        cmap, boundaries, norm = _discrete_cmap(vabs, cfg["cmap"], N_BINS)

        _map_features_white_ocean(ax)
        ax.set_extent(MAP_EXTENT, crs=proj)
        ax.set_title(cfg["title"], fontsize=13, pad=4)
        ax.text(-0.08, 1.05, label,
                transform=ax.transAxes,
                fontsize=14, fontweight="bold",
                va="top", ha="left")

        im = ax.pcolormesh(
            lon2d, lat2d, data,
            transform=proj, cmap=cmap, norm=norm,
            shading="auto", zorder=1
        )

        cb = fig.colorbar(
            mpl.cm.ScalarMappable(norm=norm, cmap=cmap),
            cax=cax1, orientation="vertical",
            extend="both", extendfrac=0.06
        )
        cb.set_label(CBAR_LABEL, fontsize=12, labelpad=12)
        cb.ax.tick_params(labelsize=10, pad=4)

    fig.savefig(savepath, dpi=plt.rcParams["savefig.dpi"], bbox_inches="tight", pad_inches=0.01)
    print(f"Saved → {savepath}")
    plt.show()
    ds.close()


# ── Entry point ───────────────────────────────────────────────────────────────
preprocess()
plot_ab()

In [ ]:
import xarray as xr
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np

# ---------------------------
# rc settings
# ---------------------------
plt.rcParams.update({
    'figure.dpi': 300,
    'savefig.dpi': 600,
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'Times New Roman', 'Times New Roman'],
    'font.size': 12,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'axes.labelpad': 0.5,
    'axes.linewidth': 1.0,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 11,
    'legend.frameon': False,
    'lines.linewidth': 1.5,
    'lines.markersize': 6,
    'axes.grid': False,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
})

INPUT_NC      = "utci_yearly_land_pct_change.nc"
OUTPUT_PNG_AB = "C:\\Users\\nerc-user\\OneDrive - Nexus365\\UTCI_NN\\Main\\figures\\supplementary\\utci_extreme_cold_heat_supplementary.png"

COASTLINE_R = "110m"
N_BINS      = 8
MAP_EXTENT  = [-180, 180, -60, 90]
CBAR_LABEL  = "Neural-UTCI and Poly-UTCI Stress Hour Difference (%)"

PANEL_CONFIG_AB = [
    dict(var="extreme_cold", title="Extreme Cold Stress (< -40 °C)", cmap="RdBu_r"),
    dict(var="extreme_heat", title="Extreme Heat Stress (> 46 °C)",  cmap="RdBu_r"),
]

def _discrete_cmap(vabs, cmap_name, n):
    boundaries = np.linspace(-vabs, vabs, n + 1)
    cmap = plt.get_cmap(cmap_name, n)
    norm = mpl.colors.BoundaryNorm(boundaries, ncolors=n)
    return cmap, boundaries, norm

def _map_features_white_ocean(ax):
    ax.set_global()
    ax.add_feature(cfeature.LAND, facecolor="whitesmoke", zorder=0)
    ax.add_feature(cfeature.OCEAN, facecolor="white", zorder=0)
    ax.coastlines(resolution=COASTLINE_R, linewidth=0.4)
    ax.add_feature(cfeature.BORDERS.with_scale(COASTLINE_R), linewidth=0.2)
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)

def plot_ab(savepath=OUTPUT_PNG_AB):
    ds = xr.open_dataset(INPUT_NC)
    lon = ds["lon"].values
    lat = ds["lat"].values
    lon2d, lat2d = np.meshgrid(lon, lat)

    proj = ccrs.Robinson(central_longitude=0)
    data_crs = ccrs.PlateCarree()

    fig = plt.figure(figsize=(12, 4))
   
    gs = fig.add_gridspec(
       2, 2,
        height_ratios=[1, 0.03],   # thinner colorbar row
        left=0.05, right=0.95,
        bottom=0.08, top=0.93,
        hspace=0.01,               # less space between plot and colorbar
        wspace=0.16
    )

    axes = [fig.add_subplot(gs[0, c], projection=proj) for c in range(2)]
    cbar_axes = [fig.add_subplot(gs[1, c]) for c in range(2)]

    panel_labels = ["a", "b"]

    for idx, (ax, cax, cfg) in enumerate(zip(axes, cbar_axes, PANEL_CONFIG_AB)):
        data = ds[cfg["var"]].values
        vabs = float(np.nanpercentile(np.abs(data), 99))
        if not np.isfinite(vabs) or vabs == 0:
            vabs = 1.0

        cmap, bounds, norm = _discrete_cmap(vabs, cfg["cmap"], N_BINS)

        _map_features_white_ocean(ax)
        ax.pcolormesh(
            lon2d, lat2d, data,
            transform=data_crs,
            cmap=cmap, norm=norm,
            shading="auto",
            zorder=1
        )
        ax.set_title(cfg["title"], fontsize=13, pad=4)
        ax.text(-0.08, 1.05, panel_labels[idx],
                transform=ax.transAxes,
                fontsize=14, fontweight="bold",
                va="top", ha="left")

        sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
        sm.set_array([])
        cb = fig.colorbar(
            sm, cax=cax, orientation="horizontal",
            boundaries=bounds,
            ticks=np.linspace(-vabs, vabs, 9),
            extend="both",
            extendfrac=0.06
        )
        cb.set_label(CBAR_LABEL, fontsize=12, labelpad=2)
        cb.ax.tick_params(labelsize=10, pad=1)
        cb.ax.xaxis.set_major_formatter(mpl.ticker.FormatStrFormatter("%.0f"))

    fig.savefig(savepath, dpi=plt.rcParams["savefig.dpi"], bbox_inches="tight", pad_inches=0.01)
    print(f"Saved → {savepath}")
    plt.show()
    ds.close()

plot_ab()

In [ ]:
import numpy as np
import xarray as xr
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

# ── Config ────────────────────────────────────────────────────────────────────
INPUT_NC     = "utci_yearly_land_pct_change.nc"
OUTPUT_PNG   = "utci_pct_density_2x3.png"

# Order: top row = heat, bottom row = cold
PANEL_CONFIG = [
    dict(var="strong_heat",      title="Heat Stress (≥ 32 °C)",             color="#d73027"),
    dict(var="very_strong_heat", title="Very Strong Heat Stress (> 38 °C)", color="#f46d43"),
    dict(var="extreme_heat",     title="Extreme Heat Stress (> 46 °C)",     color="#fdae61"),
    dict(var="cold",             title="Cold Stress (< -13 °C)",            color="#4575b4"),
    dict(var="very_cold",        title="Very Strong Cold Stress (< -27 °C)",color="#313695"),
    dict(var="extreme_cold",     title="Extreme Cold Stress (< -40 °C)",    color="#74add1"),
]

N_BINS       = 80       # histogram bins
KDE_POINTS   = 500      # resolution of KDE curve
CLIP_PCT     = 0.5      # clip top/bottom % of data to remove extreme outliers
XLABEL       = "NN − Polynomial UTCI-Stress Difference (%)"
YLABEL       = "Number of Grid Cells"


# ── Helpers ───────────────────────────────────────────────────────────────────
def _clip_data(arr, pct=CLIP_PCT):
    """Remove top and bottom pct% of values (outlier clipping)."""
    lo = np.nanpercentile(arr, pct)
    hi = np.nanpercentile(arr, 100 - pct)
    return arr[(arr >= lo) & (arr <= hi)]


# ── Plot ──────────────────────────────────────────────────────────────────────
def plot_density(savepath=OUTPUT_PNG):
    ds = xr.open_dataset(INPUT_NC)

    fig, axes = plt.subplots(
        2, 3,
        figsize=(14, 7),
        sharex=False,
        sharey=False,
    )
    fig.subplots_adjust(hspace=0.45, wspace=0.35, top=0.88, bottom=0.10)

    labels = "abcdef"

    for idx, (ax, cfg) in enumerate(zip(axes.flat, PANEL_CONFIG)):
        raw  = ds[cfg["var"]].values.ravel()
        data = _clip_data(raw[np.isfinite(raw)])

        if data.size == 0:
            ax.set_visible(False)
            continue

        color    = cfg["color"]
        lo, hi   = data.min(), data.max()

        # ── Histogram ────────────────────────────────────────────────────────
        counts, bin_edges = np.histogram(data, bins=N_BINS)
        bin_width = bin_edges[1] - bin_edges[0]
        ax.bar(
            bin_edges[:-1], counts,
            width=bin_width, align="edge",
            color=color, alpha=0.45,
            linewidth=0,
            label="Histogram",
        )

        # ── KDE (scaled to match histogram counts) ───────────────────────────
        kde      = gaussian_kde(data, bw_method="scott")
        x_kde    = np.linspace(lo, hi, KDE_POINTS)
        y_kde    = kde(x_kde) * data.size * bin_width   # scale to grid-cell counts
        ax.plot(x_kde, y_kde, color=color, linewidth=1.8, label="KDE")

        # ── Zero reference line ───────────────────────────────────────────────
        ax.axvline(0, color="black", linewidth=0.8, linestyle="--", alpha=0.6)

        # ── Median line ───────────────────────────────────────────────────────
        median = np.median(data)
        ax.axvline(median, color=color, linewidth=1.2, linestyle=":",
                   alpha=0.9, label=f"Median: {median:.1f}%")

        # ── Formatting ───────────────────────────────────────────────────────
        ax.set_title(cfg["title"], fontsize=9.5, pad=5)
        ax.set_xlabel(XLABEL, fontsize=7.5)
        ax.set_ylabel(YLABEL, fontsize=7.5)
        ax.tick_params(labelsize=7.5)
        ax.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(
            lambda x, _: f"{int(x):,}"
        ))
        ax.xaxis.set_major_formatter(mpl.ticker.FormatStrFormatter("%.0f"))

        # Panel label (a–f)
        ax.text(0.02, 0.97, labels[idx],
                transform=ax.transAxes,
                fontsize=12, fontweight="bold",
                va="top", ha="left")

        ax.legend(fontsize=7, loc="upper right", framealpha=0.6)

        # Light grid
        ax.yaxis.grid(True, linewidth=0.4, alpha=0.5, linestyle="--")
        ax.set_axisbelow(True)

        # Spine cleanup
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    fig.suptitle(
        "Distribution of % Change in UTCI-Stress Hours per Grid Cell (NN vs Polynomial, 2024)",
        fontsize=13, y=0.97,
    )

    if savepath:
        fig.savefig(savepath, dpi=150, bbox_inches="tight")
        print(f"Saved → {savepath}")

    plt.show()
    ds.close()


# ── Entry point ───────────────────────────────────────────────────────────────
plot_density()

In [ ]:
import numpy as np
import xarray as xr
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

# ── Config ────────────────────────────────────────────────────────────────────
INPUT_NC   = "utci_yearly_land_pct_change.nc"
OUTPUT_PNG = "utci_pct_density_ab_extreme.png"

PANEL_CONFIG = [
    dict(var="extreme_cold", title="Extreme Cold Stress (< −40 °C)", color="#2166ac"),
    dict(var="extreme_heat", title="Extreme Heat Stress (> 46 °C)",  color="#b2182b"),
]

N_BINS     = 80
KDE_POINTS = 500
CLIP_PCT   = 0.5
XLABEL     = "NN − Polynomial UTCI-Stress Difference (%)"
YLABEL     = "Number of Grid Cells"


# ── Helpers ───────────────────────────────────────────────────────────────────
def _clip_data(arr, pct=CLIP_PCT):
    lo = np.nanpercentile(arr, pct)
    hi = np.nanpercentile(arr, 100 - pct)
    return arr[(arr >= lo) & (arr <= hi)]


# ── Plot ──────────────────────────────────────────────────────────────────────
def plot_density_ab(savepath=OUTPUT_PNG):
    ds = xr.open_dataset(INPUT_NC)

    # Collect data and compute shared x-range
    all_data = {}
    for cfg in PANEL_CONFIG:
        raw  = ds[cfg["var"]].values.ravel()
        data = _clip_data(raw[np.isfinite(raw)])
        all_data[cfg["var"]] = data

    valid       = np.concatenate(list(all_data.values()))
    x_lo, x_hi = valid.min(), valid.max()

    fig, axes = plt.subplots(
        1, 2,
        figsize=(10, 4),
        sharex=True,
        sharey=False,
    )
    fig.subplots_adjust(wspace=0.30, top=0.88, bottom=0.15, left=0.08, right=0.97)

    for idx, (ax, cfg) in enumerate(zip(axes, PANEL_CONFIG)):
        data  = all_data[cfg["var"]]
        color = cfg["color"]

        # Histogram
        counts, bin_edges = np.histogram(data, bins=N_BINS, range=(x_lo, x_hi))
        bin_width = bin_edges[1] - bin_edges[0]
        ax.bar(
            bin_edges[:-1], counts,
            width=bin_width, align="edge",
            color=color, alpha=0.40, linewidth=0,
        )

        # KDE scaled to counts
        kde   = gaussian_kde(data, bw_method="scott")
        x_kde = np.linspace(x_lo, x_hi, KDE_POINTS)
        y_kde = kde(x_kde) * data.size * bin_width
        ax.plot(x_kde, y_kde, color=color, linewidth=2.0)

        # Zero reference
        ax.axvline(0, color="black", linewidth=0.9, linestyle="--", alpha=0.55)

        # Median
        median = np.median(data)
        ax.axvline(median, color=color, linewidth=1.3, linestyle=":", alpha=0.95)
        ax.text(
            median, ax.get_ylim()[1] * 0.93,
            f" {median:.1f}%", fontsize=8, color=color,
            va="top", ha="left" if median >= 0 else "right",
        )

        # Stats annotation box
        std = np.std(data)
        ax.text(
            0.97, 0.95,
            f"n = {data.size:,}\nmedian = {median:.1f}%\nσ = {std:.1f}%",
            transform=ax.transAxes,
            fontsize=7.5, va="top", ha="right",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white",
                      edgecolor="lightgrey", alpha=0.8),
        )

        # Panel label + title
        ax.text(-0.08, 1.08, "ab"[idx],
                transform=ax.transAxes,
                fontsize=13, fontweight="bold", va="top", ha="left")
        ax.set_title(cfg["title"], fontsize=10, pad=6)
        ax.set_xlabel(XLABEL, fontsize=8.5)
        ax.set_ylabel(YLABEL, fontsize=8.5)
        ax.tick_params(labelsize=8)
        ax.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(
            lambda x, _: f"{int(x):,}"
        ))
        ax.yaxis.grid(True, linewidth=0.35, alpha=0.5, linestyle="--")
        ax.set_axisbelow(True)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    fig.suptitle(
        "Distribution of % Change in UTCI-Stress Hours per Grid Cell (NN vs Polynomial, 2024)",
        fontsize=11, y=1.01,
    )

    if savepath:
        fig.savefig(savepath, dpi=150, bbox_inches="tight")
        print(f"Saved → {savepath}")

    plt.show()
    ds.close()


# ── Entry point ───────────────────────────────────────────────────────────────
plot_density_ab()

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import glob
import os

# Path to your monthly files
data_path = r"C:\Users\nerc-user\OneDrive - Nexus365\UTCI_NN\ERA5_exploration\data\utci_diff_mean_var"
files = sorted(glob.glob(os.path.join(data_path, "utci_diff_stats_2024-*.nc")))

# Loop over files (one per month)
for f in files:
    ds = xr.open_dataset(f)
    
    # Plot mean
    plt.figure(figsize=(12,5))
    ax = plt.axes(projection=ccrs.Robinson())
    ds['approx_diff_mean'].plot(
        ax=ax, transform=ccrs.PlateCarree(),
        cmap='coolwarm', cbar_kwargs={'label':'approx_diff mean (°C)'}
    )
    ax.coastlines()
    ax.set_title(f"Monthly UTCI Difference Mean - {os.path.basename(f)}")
    plt.show()
    
    # Plot standard deviation
    plt.figure(figsize=(12,5))
    ax = plt.axes(projection=ccrs.Robinson())
    ds['approx_diff_std'].plot(
        ax=ax, transform=ccrs.PlateCarree(),
        cmap='viridis', cbar_kwargs={'label':'approx_diff std (°C)'}
    )
    ax.coastlines()
    ax.set_title(f"Monthly UTCI Difference Std - {os.path.basename(f)}")
    plt.show()

In [ ]:
# %% [markdown]
# # Seasonal UTCI Difference Plots
# Compute mean and std for JJA (June-July-August) and DJF (Dec-Jan-Feb) and plot globally.

# %%
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import glob
import os

# Path to your monthly files
data_path = r"C:\Users\nerc-user\OneDrive - Nexus365\UTCI_NN\ERA5_exploration\data\utci_diff_mean_var"
files = sorted(glob.glob(os.path.join(data_path, "utci_diff_stats_2024-*.nc")))

# Map month number to file path
month_map = {os.path.basename(f)[21:23]: f for f in files}  # 'utci_diff_stats_2024-01.nc'

# Define seasons
seasons = {
    'JJA': ['06', '07', '08'],
    'DJF': ['12', '01', '02']
}

# Loop over seasons
for season_name, months in seasons.items():
    # Load DataArrays for mean and std
    mean_arrays = [xr.open_dataset(month_map[m])['approx_diff_mean'] for m in months]
    std_arrays  = [xr.open_dataset(month_map[m])['approx_diff_std']  for m in months]

    # Combine along new 'month' dimension
    combined_mean = xr.concat(mean_arrays, dim='month')
    combined_std  = xr.concat(std_arrays, dim='month')

    # Compute seasonal mean and seasonal std
    seasonal_mean = combined_mean.mean(dim='month')
    seasonal_std  = combined_std.std(dim='month')  # proper variability across months

    

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np

# ── paths ──────────────────────────────────────────────────────────────────
data_dir = r"C:\Users\nerc-user\OneDrive - Nexus365\UTCI_NN\ERA5_exploration\data\utci_diff_mean_var"

ds_mean = xr.open_dataset(f"{data_dir}/utci_diff_2024_mean.nc")
ds_std  = xr.open_dataset(f"{data_dir}/utci_diff_2024_std.nc")

# ── extract variables (squeeze out any size-1 time dim) ───────────────────
mean_data = ds_mean["approx_diff_mean"].squeeze()
std_data  = ds_std["approx_diff_std"].squeeze()

# ── detect lat/lon coordinate names automatically ─────────────────────────
def get_coord(ds, options):
    for name in options:
        if name in ds.coords:
            return name
    raise KeyError(f"None of {options} found in dataset coords: {list(ds.coords)}")

lat = get_coord(ds_mean, ["lat", "latitude", "y"])
lon = get_coord(ds_mean, ["lon", "longitude", "x"])

# ── plot ───────────────────────────────────────────────────────────────────
proj = ccrs.PlateCarree()
fig, axes = plt.subplots(
    1, 2, figsize=(18, 6),
    subplot_kw={"projection": proj}
)

def add_map_features(ax):
    ax.add_feature(cfeature.COASTLINE, linewidth=0.6)
    ax.add_feature(cfeature.BORDERS,   linewidth=0.3, linestyle=":")
    ax.add_feature(cfeature.LAND,      facecolor="#f0f0f0", zorder=0)
    ax.add_feature(cfeature.OCEAN,     facecolor="white", zorder=2)  # white ocean on top
    ax.gridlines(draw_labels=True, linewidth=0.3, color="gray", alpha=0.5)

# — Mean map —
ax1 = axes[0]
vmax_mean = float(np.nanpercentile(np.abs(mean_data.values), 98))
im1 = mean_data.plot(
    ax=ax1, transform=proj,
    cmap="RdBu_r",
    vmin=-vmax_mean, vmax=vmax_mean,
    add_colorbar=False
)
add_map_features(ax1)
cb1 = plt.colorbar(im1, ax=ax1, orientation="horizontal", pad=0.05, shrink=0.8)
cb1.set_label("UTCI Diff Mean (°C)", fontsize=11)
ax1.set_title("Annual Mean — approx_diff_mean (2024)", fontsize=13, fontweight="bold")

# — Std map —
ax2 = axes[1]
im2 = std_data.plot(
    ax=ax2, transform=proj,
    cmap="YlOrRd",
    vmin=0,
    add_colorbar=False
)
add_map_features(ax2)
cb2 = plt.colorbar(im2, ax=ax2, orientation="horizontal", pad=0.05, shrink=0.8)
cb2.set_label("UTCI Diff Std (°C)", fontsize=11)
ax2.set_title("Annual Std — approx_diff_std (2024)", fontsize=13, fontweight="bold")

plt.suptitle("UTCI Difference Statistics — 2024", fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(f"{data_dir}/utci_diff_2024_maps.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:

import xarray as xr
import numpy as np

data_dir = r"C:\Users\nerc-user\OneDrive - Nexus365\UTCI_NN\ERA5_exploration\data\utci_diff_mean_var"

# ── define seasons ─────────────────────────────────────────────────────────
seasons = {
    "JJA": [6, 7, 8],        # June July August
    "DJF": [12, 1, 2],       # Dec Jan Feb
}

# ── load monthly files ─────────────────────────────────────────────────────
months = [1, 2, 6, 7, 8, 12]
datasets = {}
for m in months:
    path = f"{data_dir}/utci_diff_stats_2024-{m:02d}.nc"
    datasets[m] = xr.open_dataset(path)

# ── helper: pooled std from monthly mean, std, and N ──────────────────────
def days_in_month(month):
    """Number of days per month for 2024 (leap year)."""
    days = {1:31, 2:29, 3:31, 4:30, 5:31, 6:30,
            7:31, 8:31, 9:30, 10:31, 11:30, 12:31}
    return days[month]

def pooled_stats(month_list, datasets, mean_var="approx_diff_mean", std_var="approx_diff_std"):
    """
    Compute the true pooled mean and pooled std across months.
    Uses: pooled_mean = sum(n_i * mu_i) / sum(n_i)
          pooled_var  = sum((n_i-1)*s_i^2 + n_i*(mu_i - mu_pool)^2) / (sum(n_i) - 1)
    """
    ns    = [days_in_month(m) for m in month_list]
    N     = sum(ns)

    means = [datasets[m][mean_var].squeeze() for m in month_list]
    stds  = [datasets[m][std_var].squeeze()  for m in month_list]

    # pooled mean
    pooled_mean = sum(n * mu for n, mu in zip(ns, means)) / N

    # pooled variance (accounts for both within- and between-month variance)
    pooled_var = sum(
        (n - 1) * s**2 + n * (mu - pooled_mean)**2
        for n, s, mu in zip(ns, stds, means)
    ) / (N - 1)

    pooled_std = np.sqrt(pooled_var)

    return pooled_mean, pooled_std

# ── compute and save for each season ──────────────────────────────────────
for season_name, month_list in seasons.items():
    print(f"Computing {season_name} ({month_list})...")

    pmean, pstd = pooled_stats(month_list, datasets)

    # save mean
    ds_out_mean = pmean.to_dataset(name="approx_diff_mean")
    ds_out_mean.attrs["description"] = f"Pooled mean for {season_name} 2024"
    ds_out_mean.to_netcdf(f"{data_dir}/utci_diff_{season_name}_2024_mean.nc")

    # save std
    ds_out_std = pstd.to_dataset(name="approx_diff_std")
    ds_out_std.attrs["description"] = f"Pooled std for {season_name} 2024"
    ds_out_std.to_netcdf(f"{data_dir}/utci_diff_{season_name}_2024_std.nc")

    print(f"  ✓ Saved utci_diff_{season_name}_2024_mean.nc & _std.nc")

print("Done!")

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np

# ---------------------------
# rc settings
# ---------------------------
plt.rcParams.update({
    'figure.dpi': 300,
    'savefig.dpi': 600,
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'Times New Roman', 'Times New Roman'],
    'font.size': 12,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'axes.labelpad': 0.5,
    'axes.linewidth': 1.0,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 11,
    'legend.frameon': False,
    'lines.linewidth': 1.5,
    'lines.markersize': 6,
    'axes.grid': False,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
})

data_dir = r"C:\Users\nerc-user\OneDrive - Nexus365\UTCI_NN\ERA5_exploration\data\utci_diff_mean_var"
seasons = ["JJA", "DJF"]

# load seasonal files
data = {}
for s in seasons:
    data[s] = {
        "mean": xr.open_dataset(f"{data_dir}/utci_diff_{s}_2024_mean.nc")["approx_diff_mean"].squeeze(),
        "std":  xr.open_dataset(f"{data_dir}/utci_diff_{s}_2024_std.nc")["approx_diff_std"].squeeze(),
    }

# shared colormap limits
all_means = np.concatenate([data[s]["mean"].values.ravel() for s in seasons])
all_stds  = np.concatenate([data[s]["std"].values.ravel()  for s in seasons])

vmax_mean = float(np.nanpercentile(np.abs(all_means), 99.9))
vmax_std  = float(np.nanpercentile(all_stds, 99.9))

# Robinson projection for display, PlateCarree for data
proj = ccrs.Robinson(central_longitude=0)
data_crs = ccrs.PlateCarree()

# figure/layout 
fig = plt.figure(figsize=(12, 8))
gs = fig.add_gridspec(
    2, 2,
    left=0.04, right=0.88,
    bottom=0.04, top=0.97,
    hspace=0.00,
    wspace=0.05
)

axes = np.empty((2, 2), dtype=object)
for r in range(2):
    for c in range(2):
        axes[r, c] = fig.add_subplot(gs[r, c], projection=proj)

season_labels = {"JJA": "June–July–August", "DJF": "December–January–February"}

def add_map_features(ax):
    ax.set_global()
    ax.add_feature(cfeature.LAND, facecolor="#f0f0f0", zorder=0)
    ax.add_feature(cfeature.OCEAN, facecolor="white", zorder=1)
    ax.coastlines(linewidth=0.6)
    ax.add_feature(cfeature.BORDERS, linewidth=0.3, linestyle=":")
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)

# plot maps
im_mean = None
im_std = None

for col, season in enumerate(seasons):
    # top row: mean
    ax = axes[0, col]
    im_mean = data[season]["mean"].plot(
        ax=ax, transform=data_crs,
        cmap="RdBu_r",
        vmin=-vmax_mean, vmax=vmax_mean,
        add_colorbar=False
    )
    add_map_features(ax)
    ax.set_title(f"{season}", fontsize=13)
    ax.set_xlabel("")
    ax.set_ylabel("")

    # bottom row: std
    ax = axes[1, col]
    im_std = data[season]["std"].plot(
        ax=ax, transform=data_crs,
        cmap="YlOrRd",
        vmin=0, vmax=vmax_std,
        add_colorbar=False
    )
    add_map_features(ax)
    ax.set_title("")  # no title on second row
    ax.set_xlabel("")
    ax.set_ylabel("")

# optional panel labels
panel_axes = [axes[0, 0], axes[0, 1], axes[1, 0], axes[1, 1]]
panel_labels = ["a", "b", "c", "d"]
for ax, label in zip(panel_axes, panel_labels):
    ax.text(-0.08, 1.05, label,
            transform=ax.transAxes,
            fontsize=14,
            fontweight="bold",
            va="top",
            ha="left")

# shared vertical colorbar for the top row (mean)
cax1 = fig.add_axes([0.90, 0.56, 0.010, 0.31])
cb1 = fig.colorbar(
    im_mean, cax=cax1, orientation="vertical",
    extend="both", extendfrac=0.06,
    ticks=np.arange(-np.ceil(vmax_mean), np.ceil(vmax_mean) + 1, 1)
)
cb1.set_label("Approximator Mean (°C)", fontsize=12, labelpad=12)
cb1.ax.tick_params(labelsize=10, pad=4)

# shared vertical colorbar for the bottom row (std)
cax2 = fig.add_axes([0.90, 0.13, 0.010, 0.31])
cb2 = fig.colorbar(
    im_std, cax=cax2, orientation="vertical",
    extend="max", extendfrac=0.06,
    ticks=np.arange(0, np.ceil(vmax_std) + 1, 1)
)
cb2.set_label("Approximator Standard Deviation (°C)", fontsize=12, labelpad=12)
cb2.ax.tick_params(labelsize=10, pad=4)

fig.savefig(
    f"C://Users/nerc-user/OneDrive - Nexus365/UTCI_NN/Main/figures/supplementary/utci_diff_seasonal_2024_maps_robinson.png",
    dpi=600,
    bbox_inches="tight",
    pad_inches=0.01
)
plt.show()

In [ ]:
"""
utci_six_panel_figure.py
========================
Publication-ready 6-panel figure (3 rows × 2 columns) for Nature submission.

  Row 1 (a, b) : Annual mean & std of UTCI approximation difference
  Row 2 (c, d) : Yearly stress-hours difference – Very Strong Cold / Very Strong Heat
  Row 3 (e, f) : KDE density distributions – Extreme Cold / Extreme Heat % change

Adjust the three DATA PATHS blocks below to match your file locations,
then run:  python utci_six_panel_figure.py
"""

import os
import numpy as np
import xarray as xr
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from scipy.stats import gaussian_kde

# ═══════════════════════════════════════════════════════════════════════════════
# DATA PATHS  ── edit these three blocks ────────────────────────────────────────
# ═══════════════════════════════════════════════════════════════════════════════

# Row 1 – mean / std maps
MEAN_STD_DIR = (
    r"C:\Users\nerc-user\OneDrive - Nexus365"
    r"\UTCI_NN\ERA5_exploration\data\utci_diff_mean_var"
)
MEAN_NC = os.path.join(MEAN_STD_DIR, "utci_diff_2024_mean.nc")
STD_NC  = os.path.join(MEAN_STD_DIR, "utci_diff_2024_std.nc")

# Row 2 – stress-hours maps
HOURS_NC = "utci_yearly_land_hours.nc"

# Row 3 – % change distributions
PCT_NC = "utci_yearly_land_pct_change.nc"

# Output
OUTPUT_PNG = "utci_six_panel_nature.png"

# ═══════════════════════════════════════════════════════════════════════════════
# GLOBAL STYLE  (Nature guidelines: 7 pt minimum, Helvetica/Arial)
# ═══════════════════════════════════════════════════════════════════════════════

mpl.rcParams.update({
    "font.family":       "sans-serif",
    "font.sans-serif":   ["Helvetica", "Arial", "DejaVu Sans"],
    "font.size":         7,
    "axes.titlesize":    8,
    "axes.labelsize":    7,
    "xtick.labelsize":   6,
    "ytick.labelsize":   6,
    "axes.linewidth":    0.5,
    "xtick.major.width": 0.5,
    "ytick.major.width": 0.5,
    "lines.linewidth":   0.8,
    "patch.linewidth":   0.5,
    "savefig.dpi":       300,
    "figure.dpi":        100,
})

PANEL_LABELS = list("abcdef")

# ═══════════════════════════════════════════════════════════════════════════════
# SHARED HELPERS
# ═══════════════════════════════════════════════════════════════════════════════

MAP_EXTENT   = [-180, 180, -60, 90]
COASTLINE_RES = "110m"


def _add_map_features(ax):
    """Minimal, clean map background suitable for Nature."""
    ax.add_feature(
        cfeature.OCEAN.with_scale(COASTLINE_RES),
        facecolor="white", zorder=0,
    )
    ax.add_feature(
        cfeature.LAND.with_scale(COASTLINE_RES),
        facecolor="#f4f4f4", zorder=0,
    )
    ax.add_feature(
        cfeature.COASTLINE.with_scale(COASTLINE_RES),
        linewidth=0.35, edgecolor="#444444", zorder=2,
    )
    ax.add_feature(
        cfeature.BORDERS.with_scale(COASTLINE_RES),
        linewidth=0.18, linestyle=":", edgecolor="#888888", zorder=2,
    )
    ax.set_extent(MAP_EXTENT, crs=ccrs.PlateCarree())


def _get_coord(ds, options):
    for name in options:
        if name in ds.coords:
            return name
    raise KeyError(f"None of {options} found in {list(ds.coords)}")


def _discrete_cmap(vabs, cmap_name, n=13):
    boundaries = np.linspace(-vabs, vabs, n + 1)
    cmap = plt.get_cmap(cmap_name, n)
    norm = mpl.colors.BoundaryNorm(boundaries, ncolors=n)
    return cmap, boundaries, norm


def _green_pink_cmap(n=13):
    """Diverging green (negative) → white (zero) → pink (positive) colormap."""
    from matplotlib.colors import LinearSegmentedColormap
    colors = [
        "#1a7a4a",   # deep green  (most negative)
        "#4dac58",
        "#80c97a",
        "#b3e2a0",
        "#d9f0c8",
        "#ffffff",   # white (zero)
        "#fcd5e0",
        "#f9aabb",
        "#f47fa0",
        "#e04c7a",
        "#c0235a",   # deep pink   (most positive)
    ]
    cmap = LinearSegmentedColormap.from_list("green_pink", colors, N=n)
    return cmap


def _add_panel_label(ax, label, x=-0.06, y=1.04):
    ax.text(
        x, y, label,
        transform=ax.transAxes,
        fontsize=9, fontweight="bold",
        va="top", ha="left",
    )


def _horizontal_colorbar(fig, mappable, ax_ref, label,
                          pad=0.01, height=0.012, extend="both"):
    """Place a slim horizontal colorbar below ax_ref."""
    pos = ax_ref.get_position()
    cax = fig.add_axes([
        pos.x0,
        pos.y0 - pad - height,
        pos.width,
        height,
    ])
    cb = fig.colorbar(mappable, cax=cax, orientation="horizontal",
                      extend=extend)
    cb.set_label(label, fontsize=6.5, labelpad=2)
    cb.ax.tick_params(labelsize=5.5, rotation=30, length=2, width=0.4)
    return cb


# ═══════════════════════════════════════════════════════════════════════════════
# ROW 1 – mean & std maps
# ═══════════════════════════════════════════════════════════════════════════════

def _draw_row1(fig, ax_a, ax_b):
    ds_mean = xr.open_dataset(MEAN_NC)
    ds_std  = xr.open_dataset(STD_NC)

    lat_name = _get_coord(ds_mean, ["lat", "latitude", "y"])
    lon_name = _get_coord(ds_mean, ["lon", "longitude", "x"])

    mean_data = ds_mean["approx_diff_mean"].squeeze()
    std_data  = ds_std["approx_diff_std"].squeeze()

    proj = ccrs.PlateCarree()

    # ── panel a: mean ──────────────────────────────────────────────────────────
    vmax = float(np.nanpercentile(np.abs(mean_data.values), 98))
    norm_mean = mpl.colors.TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
    im_a = ax_a.pcolormesh(
        ds_mean[lon_name].values, ds_mean[lat_name].values,
        mean_data.values,
        transform=proj, cmap="RdBu_r", norm=norm_mean,
        shading="auto", zorder=1,
    )
    _add_map_features(ax_a)
    ax_a.set_title("Annual mean UTCI difference (NN − Polynomial)", pad=3)
    _add_panel_label(ax_a, "a")
    _horizontal_colorbar(fig, im_a, ax_a, "Mean UTCI difference (°C)")

    # ── panel b: std ───────────────────────────────────────────────────────────
    vmax_std = float(np.nanpercentile(std_data.values[np.isfinite(std_data.values)], 98))
    norm_std = mpl.colors.Normalize(vmin=0, vmax=vmax_std)
    im_b = ax_b.pcolormesh(
        ds_std[lon_name].values, ds_std[lat_name].values,
        std_data.values,
        transform=proj, cmap="YlOrRd", norm=norm_std,
        shading="auto", zorder=1,
    )
    _add_map_features(ax_b)
    ax_b.set_title("Annual std of UTCI difference (NN − Polynomial)", pad=3)
    _add_panel_label(ax_b, "b")
    _horizontal_colorbar(fig, im_b, ax_b, "Std of UTCI difference (°C)", extend="max")

    ds_mean.close(); ds_std.close()


# ═══════════════════════════════════════════════════════════════════════════════
# ROW 2 – stress-hours maps
# ═══════════════════════════════════════════════════════════════════════════════

HOURS_PANEL_CONFIG = [
    dict(var="very_cold",        title="Very Strong Cold Stress hours (< \u221227 \u00b0C)"),
    dict(var="very_strong_heat", title="Very Strong Heat Stress hours (> 38 \u00b0C)"),
]


def _draw_row2(fig, ax_c, ax_d):
    ds   = xr.open_dataset(HOURS_NC)
    lon  = ds["lon"].values
    lat  = ds["lat"].values
    lon2d, lat2d = np.meshgrid(lon, lat)
    proj = ccrs.PlateCarree()

    for ax, cfg, label in zip([ax_c, ax_d], HOURS_PANEL_CONFIG, ["c", "d"]):
        data = ds[cfg["var"]].values
        vabs = float(np.nanpercentile(np.abs(data[np.isfinite(data)]), 99))
        if not np.isfinite(vabs) or vabs == 0:
            vabs = 1.0

        n          = 13
        cmap       = _green_pink_cmap(n)
        boundaries = np.linspace(-vabs, vabs, n + 1)
        norm       = mpl.colors.BoundaryNorm(boundaries, ncolors=n)

        ax.pcolormesh(
            lon2d, lat2d, data,
            transform=proj, cmap=cmap, norm=norm,
            shading="auto", zorder=1,
        )
        _add_map_features(ax)
        ax.set_title(cfg["title"], pad=3)
        _add_panel_label(ax, label)

        cb = _horizontal_colorbar(
            fig,
            mpl.cm.ScalarMappable(norm=norm, cmap=cmap),
            ax,
            "NN \u2212 Polynomial stress-hours difference (h)",
        )
        cb.set_ticks(boundaries[::2])
        cb.ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f"))

    ds.close()


# ═══════════════════════════════════════════════════════════════════════════════
# ROW 3 – KDE density distributions
# ═══════════════════════════════════════════════════════════════════════════════

DIST_PANEL_CONFIG = [
    dict(var="extreme_cold", title="Extreme Cold Stress (< −40 °C)", color="#2166ac"),
    dict(var="extreme_heat", title="Extreme Heat Stress (> 46 °C)",  color="#b2182b"),
]

N_BINS_HIST  = 80
KDE_POINTS   = 500
CLIP_PCT     = 0.5


def _clip_data(arr, pct=CLIP_PCT):
    lo = np.nanpercentile(arr, pct)
    hi = np.nanpercentile(arr, 100 - pct)
    return arr[(arr >= lo) & (arr <= hi)]


def _draw_row3(ax_e, ax_f):
    ds = xr.open_dataset(PCT_NC)

    all_data = {}
    for cfg in DIST_PANEL_CONFIG:
        raw  = ds[cfg["var"]].values.ravel()
        data = _clip_data(raw[np.isfinite(raw)])
        all_data[cfg["var"]] = data

    valid       = np.concatenate(list(all_data.values()))
    x_lo, x_hi = valid.min(), valid.max()

    for ax, cfg, label in zip([ax_e, ax_f], DIST_PANEL_CONFIG, ["e", "f"]):
        data  = all_data[cfg["var"]]
        color = cfg["color"]

        # Histogram bars
        counts, edges = np.histogram(data, bins=N_BINS_HIST, range=(x_lo, x_hi))
        bw = edges[1] - edges[0]
        ax.bar(edges[:-1], counts, width=bw, align="edge",
               color=color, alpha=0.35, linewidth=0)

        # KDE curve scaled to counts
        kde   = gaussian_kde(data, bw_method="scott")
        x_kde = np.linspace(x_lo, x_hi, KDE_POINTS)
        y_kde = kde(x_kde) * data.size * bw
        ax.plot(x_kde, y_kde, color=color, linewidth=1.4)

        # Reference lines
        ax.axvline(0,      color="black", linewidth=0.7,
                   linestyle="--", alpha=0.55, zorder=3)
        median = np.median(data)
        ax.axvline(median, color=color,   linewidth=1.0,
                   linestyle=":",  alpha=0.9,  zorder=3)

        # Median label on plot
        ylim_top = ax.get_ylim()[1]
        ax.text(
            median, ylim_top * 0.93,
            f" {median:.1f}%",
            fontsize=6, color=color,
            va="top", ha="left" if median >= 0 else "right",
        )

        # Print stats to console (no box on figure)
        print(
            f"  Panel {label} – {cfg['title']}\n"
            f"    n      = {data.size:,}\n"
            f"    mean   = {np.mean(data):.2f}%\n"
            f"    median = {median:.2f}%\n"
            f"    std    = {np.std(data):.2f}%\n"
            f"    min    = {data.min():.2f}%\n"
            f"    max    = {data.max():.2f}%\n"
        )

        _add_panel_label(ax, label, x=-0.09, y=1.06)
        ax.set_title(cfg["title"], pad=3)
        ax.set_xlabel("NN − Polynomial UTCI-stress difference (%)", labelpad=2)
        ax.set_ylabel("Number of grid cells", labelpad=2)
        ax.yaxis.set_major_formatter(
            mticker.FuncFormatter(lambda x, _: f"{int(x):,}")
        )
        ax.yaxis.grid(True, linewidth=0.3, alpha=0.4, linestyle="--")
        ax.set_axisbelow(True)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.tick_params(axis="both", length=2, width=0.4)

    ds.close()


# ═══════════════════════════════════════════════════════════════════════════════
# ASSEMBLE FIGURE
# ═══════════════════════════════════════════════════════════════════════════════

def build_figure():
    proj = ccrs.PlateCarree()

    # Nature single-column = 89 mm wide; double-column = 183 mm wide.
    # At 300 dpi, 183 mm ≈ 7.2 inches.  Height chosen for 3 rows with maps.
    fig = plt.figure(figsize=(7.2, 9.6))

    # ── GridSpec: 3 rows, 2 cols ─────────────────────────────────────────────
    # Rows 1 & 2 are maps (taller), row 3 is histogram (shorter).
    # Extra vertical space at the bottom of each map row is used for colourbars.
    gs = mpl.gridspec.GridSpec(
        3, 2,
        figure=fig,
        height_ratios=[1.0, 1.0, 0.75],
        hspace=0.52,   # vertical gap between rows (room for colourbars + titles)
        wspace=0.10,   # horizontal gap between columns
        left=0.06, right=0.97,
        top=0.96, bottom=0.07,
    )

    ax_a = fig.add_subplot(gs[0, 0], projection=proj)
    ax_b = fig.add_subplot(gs[0, 1], projection=proj)
    ax_c = fig.add_subplot(gs[1, 0], projection=proj)
    ax_d = fig.add_subplot(gs[1, 1], projection=proj)
    ax_e = fig.add_subplot(gs[2, 0])
    ax_f = fig.add_subplot(gs[2, 1])

    # Fix map aspect ratio for all cartopy axes
    for ax in [ax_a, ax_b, ax_c, ax_d]:
        ax.set_aspect("auto")

    print("Drawing row 1 (mean & std) …")
    _draw_row1(fig, ax_a, ax_b)

    print("Drawing row 2 (stress hours) …")
    _draw_row2(fig, ax_c, ax_d)

    print("Drawing row 3 (distributions) …")
    _draw_row3(ax_e, ax_f)

    fig.savefig(OUTPUT_PNG, dpi=300, bbox_inches="tight", facecolor="white")
    print(f"\n✓  Saved → {OUTPUT_PNG}")
    plt.show()


# ═══════════════════════════════════════════════════════════════════════════════
if __name__ == "__main__":
    build_figure()

In [ ]:
"""
utci_six_panel_figure_robinson.py
=================================
Publication-ready 6-panel figure (3 rows × 2 columns) for Nature submission.

Row 1 (a, b) : Annual mean & std of UTCI approximation difference
Row 2 (c, d) : Yearly stress-hours difference – Very Strong Cold / Very Strong Heat
Row 3 (e, f) : KDE density distributions – Extreme Cold / Extreme Heat % change

Changes from the original:
- Robinson projection for all map panels
- Thinner horizontal colorbars
- Less space between maps and colorbars
- Same overall clean Nature-style layout
"""

import os
import numpy as np
import xarray as xr
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from scipy.stats import gaussian_kde

# ═══════════════════════════════════════════════════════════════════════════════
# DATA PATHS  ── edit these blocks to match your files
# ═══════════════════════════════════════════════════════════════════════════════

# Row 1 – mean / std maps
MEAN_STD_DIR = (
    r"C:\Users\nerc-user\OneDrive - Nexus365"
    r"\UTCI_NN\ERA5_exploration\data\utci_diff_mean_var"
)
MEAN_NC = os.path.join(MEAN_STD_DIR, "utci_diff_2024_mean.nc")
STD_NC  = os.path.join(MEAN_STD_DIR, "utci_diff_2024_std.nc")

# Row 2 – stress-hours maps
HOURS_NC = "utci_yearly_land_hours.nc"

# Row 3 – % change distributions
PCT_NC = "utci_yearly_land_pct_change.nc"

# Output
OUTPUT_PNG = "utci_six_panel_robinson.png"

# ═══════════════════════════════════════════════════════════════════════════════
# GLOBAL STYLE
# ═══════════════════════════════════════════════════════════════════════════════

mpl.rcParams.update({
    "font.family":       "sans-serif",
    "font.sans-serif":   ["Helvetica", "Arial", "DejaVu Sans"],
    "font.size":         7,
    "axes.titlesize":    8,
    "axes.labelsize":    7,
    "xtick.labelsize":   6,
    "ytick.labelsize":   6,
    "axes.linewidth":    0.5,
    "xtick.major.width": 0.5,
    "ytick.major.width": 0.5,
    "lines.linewidth":   0.8,
    "patch.linewidth":   0.5,
    "savefig.dpi":       300,
    "figure.dpi":        100,
})

PANEL_LABELS = list("abcdef")

# ═══════════════════════════════════════════════════════════════════════════════
# SHARED HELPERS
# ═══════════════════════════════════════════════════════════════════════════════

MAP_EXTENT = [-180, 180, -60, 90]
COASTLINE_RES = "110m"

# Robinson for map panels
MAP_PROJ = ccrs.Robinson(central_longitude=0)
DATA_PROJ = ccrs.PlateCarree()


def _add_map_features(ax):
    """Minimal, clean map background suitable for Nature."""
    ax.set_global()
    ax.add_feature(
        cfeature.OCEAN.with_scale(COASTLINE_RES),
        facecolor="white", zorder=0,
    )
    ax.add_feature(
        cfeature.LAND.with_scale(COASTLINE_RES),
        facecolor="#f4f4f4", zorder=0,
    )
    ax.add_feature(
        cfeature.COASTLINE.with_scale(COASTLINE_RES),
        linewidth=0.35, edgecolor="#444444", zorder=2,
    )
    ax.add_feature(
        cfeature.BORDERS.with_scale(COASTLINE_RES),
        linewidth=0.18, linestyle=":", edgecolor="#888888", zorder=2,
    )


def _get_coord(ds, options):
    for name in options:
        if name in ds.coords:
            return name
    raise KeyError(f"None of {options} found in {list(ds.coords)}")


def _discrete_cmap(vabs, cmap_name, n=13):
    boundaries = np.linspace(-vabs, vabs, n + 1)
    cmap = plt.get_cmap(cmap_name, n)
    norm = mpl.colors.BoundaryNorm(boundaries, ncolors=n)
    return cmap, boundaries, norm


def _green_pink_cmap(n=13):
    """Diverging green (negative) → white (zero) → pink (positive) colormap."""
    from matplotlib.colors import LinearSegmentedColormap
    colors = [
        "#1a7a4a",   # deep green
        "#4dac58",
        "#80c97a",
        "#b3e2a0",
        "#d9f0c8",
        "#ffffff",   # white
        "#fcd5e0",
        "#f9aabb",
        "#f47fa0",
        "#e04c7a",
        "#c0235a",   # deep pink
    ]
    cmap = LinearSegmentedColormap.from_list("green_pink", colors, N=n)
    return cmap


def _add_panel_label(ax, label, x=-0.06, y=1.04):
    ax.text(
        x, y, label,
        transform=ax.transAxes,
        fontsize=9, fontweight="bold",
        va="top", ha="left",
    )


def _horizontal_colorbar(fig, mappable, ax_ref, label,
                         pad=0.004, height=0.008, extend="both"):
    """Place a slim horizontal colorbar below ax_ref."""
    pos = ax_ref.get_position()
    cax = fig.add_axes([
        pos.x0,
        pos.y0 - pad - height,
        pos.width,
        height,
    ])
    cb = fig.colorbar(
        mappable, cax=cax, orientation="horizontal", extend=extend
    )
    cb.set_label(label, fontsize=6.5, labelpad=1.5)
    cb.ax.tick_params(labelsize=5.5, rotation=0, length=2, width=0.4, pad=1)
    return cb


# ═══════════════════════════════════════════════════════════════════════════════
# ROW 1 – mean & std maps
# ═══════════════════════════════════════════════════════════════════════════════

def _draw_row1(fig, ax_a, ax_b):
    ds_mean = xr.open_dataset(MEAN_NC)
    ds_std  = xr.open_dataset(STD_NC)

    lat_name = _get_coord(ds_mean, ["lat", "latitude", "y"])
    lon_name = _get_coord(ds_mean, ["lon", "longitude", "x"])

    mean_data = ds_mean["approx_diff_mean"].squeeze()
    std_data  = ds_std["approx_diff_std"].squeeze()

    proj = DATA_PROJ

    # panel a: mean
    vmax = float(np.nanpercentile(np.abs(mean_data.values), 98))
    norm_mean = mpl.colors.TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
    im_a = ax_a.pcolormesh(
        ds_mean[lon_name].values, ds_mean[lat_name].values,
        mean_data.values,
        transform=proj, cmap="RdBu_r", norm=norm_mean,
        shading="auto", zorder=1,
    )
    _add_map_features(ax_a)
    ax_a.set_title("Annual mean UTCI difference (Neural-UTCI − Poly-UTCI)", pad=3)
    _add_panel_label(ax_a, "a")
    _horizontal_colorbar(fig, im_a, ax_a, "Mean UTCI difference (°C)")

    # panel b: std
    finite_std = std_data.values[np.isfinite(std_data.values)]
    vmax_std = float(np.nanpercentile(finite_std, 98))
    norm_std = mpl.colors.Normalize(vmin=0, vmax=vmax_std)
    im_b = ax_b.pcolormesh(
        ds_std[lon_name].values, ds_std[lat_name].values,
        std_data.values,
        transform=proj, cmap="YlOrRd", norm=norm_std,
        shading="auto", zorder=1,
    )
    _add_map_features(ax_b)
    ax_b.set_title("Annual std of UTCI difference (Neural-UTCI − Poly-UTCI)", pad=3)
    _add_panel_label(ax_b, "b")
    _horizontal_colorbar(
        fig, im_b, ax_b, "Std of UTCI difference (°C)", extend="max"
    )

    ds_mean.close()
    ds_std.close()


# ═══════════════════════════════════════════════════════════════════════════════
# ROW 2 – stress-hours maps
# ═══════════════════════════════════════════════════════════════════════════════

HOURS_PANEL_CONFIG = [
    dict(var="very_cold",        title="Very Strong Cold Stress hours (< −27 °C)"),
    dict(var="very_strong_heat", title="Very Strong Heat Stress hours (> 38 °C)"),
]


def _draw_row2(fig, ax_c, ax_d):
    ds = xr.open_dataset(HOURS_NC)
    lon = ds["lon"].values
    lat = ds["lat"].values
    lon2d, lat2d = np.meshgrid(lon, lat)
    proj = DATA_PROJ

    for ax, cfg, label in zip([ax_c, ax_d], HOURS_PANEL_CONFIG, ["c", "d"]):
        data = ds[cfg["var"]].values
        finite = data[np.isfinite(data)]
        vabs = float(np.nanpercentile(np.abs(finite), 99))
        if not np.isfinite(vabs) or vabs == 0:
            vabs = 1.0

        n = 13
        cmap = _green_pink_cmap(n)
        boundaries = np.linspace(-vabs, vabs, n + 1)
        norm = mpl.colors.BoundaryNorm(boundaries, ncolors=n)

        ax.pcolormesh(
            lon2d, lat2d, data,
            transform=proj, cmap=cmap, norm=norm,
            shading="auto", zorder=1,
        )
        _add_map_features(ax)
        ax.set_title(cfg["title"], pad=3)
        _add_panel_label(ax, label)

        cb = _horizontal_colorbar(
            fig,
            mpl.cm.ScalarMappable(norm=norm, cmap=cmap),
            ax,
            "Neural-UTCI − Poly-UTCI stress-hours difference (h)",
        )
        cb.set_ticks(boundaries[::2])
        cb.ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f"))

    ds.close()


# ═══════════════════════════════════════════════════════════════════════════════
# ROW 3 – KDE density distributions
# ════=============================================================================

DIST_PANEL_CONFIG = [
    dict(var="extreme_cold", title="Extreme Cold Stress (< −40 °C)", color="#2166ac"),
    dict(var="extreme_heat", title="Extreme Heat Stress (> 46 °C)",  color="#b2182b"),
]

N_BINS_HIST = 80
KDE_POINTS  = 500
CLIP_PCT    = 0.5


def _clip_data(arr, pct=CLIP_PCT):
    lo = np.nanpercentile(arr, pct)
    hi = np.nanpercentile(arr, 100 - pct)
    return arr[(arr >= lo) & (arr <= hi)]


def _draw_row3(ax_e, ax_f):
    ds = xr.open_dataset(PCT_NC)

    all_data = {}
    for cfg in DIST_PANEL_CONFIG:
        raw = ds[cfg["var"]].values.ravel()
        data = _clip_data(raw[np.isfinite(raw)])
        all_data[cfg["var"]] = data

    valid = np.concatenate(list(all_data.values()))
    x_lo, x_hi = valid.min(), valid.max()

    for ax, cfg, label in zip([ax_e, ax_f], DIST_PANEL_CONFIG, ["e", "f"]):
        data = all_data[cfg["var"]]
        color = cfg["color"]

        # Histogram bars
        counts, edges = np.histogram(data, bins=N_BINS_HIST, range=(x_lo, x_hi))
        bw = edges[1] - edges[0]
        ax.bar(
            edges[:-1], counts, width=bw, align="edge",
            color=color, alpha=0.35, linewidth=0
        )

        # KDE curve scaled to counts
        kde = gaussian_kde(data, bw_method="scott")
        x_kde = np.linspace(x_lo, x_hi, KDE_POINTS)
        y_kde = kde(x_kde) * data.size * bw
        ax.plot(x_kde, y_kde, color=color, linewidth=1.4)

        # Reference lines
        ax.axvline(0, color="black", linewidth=0.7,
                   linestyle="--", alpha=0.55, zorder=3)
        median = np.median(data)
        ax.axvline(median, color=color, linewidth=1.0,
                   linestyle=":", alpha=0.9, zorder=3)

        # Median label on plot
        ylim_top = ax.get_ylim()[1]
        ax.text(
            median, ylim_top * 0.93,
            f" {median:.1f}%",
            fontsize=6, color=color,
            va="top", ha="left" if median >= 0 else "right",
        )

        print(
            f"  Panel {label} – {cfg['title']}\n"
            f"    n      = {data.size:,}\n"
            f"    mean   = {np.mean(data):.2f}%\n"
            f"    median = {median:.2f}%\n"
            f"    std    = {np.std(data):.2f}%\n"
            f"    min    = {data.min():.2f}%\n"
            f"    max    = {data.max():.2f}%\n"
        )

        _add_panel_label(ax, label, x=-0.09, y=1.06)
        ax.set_title(cfg["title"], pad=3)
        ax.set_xlabel("Neural-UTCI − Poly-UTCI UTCI-stress difference (%)", labelpad=2)
        ax.set_ylabel("Number of grid cells", labelpad=2)
        ax.yaxis.set_major_formatter(
            mticker.FuncFormatter(lambda x, _: f"{int(x):,}")
        )
        ax.yaxis.grid(True, linewidth=0.3, alpha=0.4, linestyle="--")
        ax.set_axisbelow(True)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.tick_params(axis="both", length=2, width=0.4)

    ds.close()


# ═══════════════════════════════════════════════════════════════════════════════
# ASSEMBLE FIGURE
# ═══════════════════════════════════════════════════════════════════════════════

def build_figure():
    fig = plt.figure(figsize=(7.2, 9.6))

    # 3 rows, 2 columns
    gs = mpl.gridspec.GridSpec(
        3, 2,
        figure=fig,
        height_ratios=[1.0, 1.0, 0.75],
        hspace=0.40,
        wspace=0.10,
        left=0.06, right=0.97,
        top=0.96, bottom=0.07,
    )

    ax_a = fig.add_subplot(gs[0, 0], projection=MAP_PROJ)
    ax_b = fig.add_subplot(gs[0, 1], projection=MAP_PROJ)
    ax_c = fig.add_subplot(gs[1, 0], projection=MAP_PROJ)
    ax_d = fig.add_subplot(gs[1, 1], projection=MAP_PROJ)
    ax_e = fig.add_subplot(gs[2, 0])
    ax_f = fig.add_subplot(gs[2, 1])

    # Let cartopy axes fill the subplot area cleanly
    for ax in [ax_a, ax_b, ax_c, ax_d]:
        ax.set_aspect("auto")

    print("Drawing row 1 (mean & std) …")
    _draw_row1(fig, ax_a, ax_b)

    print("Drawing row 2 (stress hours) …")
    _draw_row2(fig, ax_c, ax_d)

    print("Drawing row 3 (distributions) …")
    _draw_row3(ax_e, ax_f)

    fig.savefig(OUTPUT_PNG, dpi=300, bbox_inches="tight", facecolor="white")
    print(f"\n✓  Saved → {OUTPUT_PNG}")
    plt.show()


# ═══════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    build_figure()

In [ ]:
"""
utci_six_panel_robinson.py
==========================
Publication-ready 6-panel figure (3 rows × 2 columns) for Nature submission.

  Row 1 (a, b) : Annual mean & std of UTCI approximation difference
  Row 2 (c, d) : Yearly stress-hours difference – Very Strong Cold / Very Strong Heat
  Row 3 (e, f) : KDE density distributions – Extreme Cold / Extreme Heat % change

Map panels are rendered on a Robinson projection.
Adjust the three DATA PATHS blocks below to match your file locations,
then run:  python utci_six_panel_robinson.py
"""

import os
import numpy as np
import xarray as xr
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.util import add_cyclic_point
from scipy.stats import gaussian_kde

# ═══════════════════════════════════════════════════════════════════════════════
# DATA PATHS  ── edit these three blocks ────────────────────────────────────────
# ═══════════════════════════════════════════════════════════════════════════════

# Row 1 – mean / std maps
MEAN_STD_DIR = (
    r"C:\Users\nerc-user\OneDrive - Nexus365"
    r"\UTCI_NN\ERA5_exploration\data\utci_diff_mean_var"
)
MEAN_NC = os.path.join(MEAN_STD_DIR, "utci_diff_2024_mean.nc")
STD_NC  = os.path.join(MEAN_STD_DIR, "utci_diff_2024_std.nc")

# Row 2 – stress-hours maps
HOURS_NC = "utci_yearly_land_hours.nc"

# Row 3 – % change distributions
PCT_NC = "utci_yearly_land_pct_change.nc"

# Output
OUTPUT_PNG = "utci_six_panel_robinson.png"

# ═══════════════════════════════════════════════════════════════════════════════
# GLOBAL STYLE  (Nature guidelines: 7 pt minimum, Helvetica/Arial)
# ═══════════════════════════════════════════════════════════════════════════════

mpl.rcParams.update({
    "figure.dpi":        300,
    "savefig.dpi":       600,
    "font.family":       "serif",
    "font.serif":        ["Times New Roman", "Times", "DejaVu Serif"],
    "font.size":         12,
    "axes.titlesize":    13,
    "axes.labelsize":    12,
    "axes.linewidth":    1.0,
    "xtick.labelsize":   11,
    "ytick.labelsize":   11,
    "legend.fontsize":   11,
    "legend.frameon":    False,
    "lines.linewidth":   1.5,
    "lines.markersize":  6,
    "axes.grid":         False,
    "pdf.fonttype":      42,
    "ps.fonttype":       42,
    "xtick.major.width": 1.0,
    "ytick.major.width": 1.0,
    "patch.linewidth":   1.0,
})

PANEL_LABELS = list("abcdef")

# ═══════════════════════════════════════════════════════════════════════════════
# SHARED HELPERS
# ═══════════════════════════════════════════════════════════════════════════════

MAP_EXTENT = [-180, 180, -60, 90]
COASTLINE_RES = "110m"
MAP_PROJ = ccrs.Robinson(central_longitude=0)
DATA_PROJ = ccrs.PlateCarree()


def _add_map_features(ax, gridlines=False):
    """Map background for Robinson projection figures."""
    ax.set_global()
    ax.add_feature(
        cfeature.LAND.with_scale(COASTLINE_RES),
        facecolor="#f4f4f4", zorder=0,
    )
    ax.add_feature(
        cfeature.OCEAN.with_scale(COASTLINE_RES),
        facecolor="white", zorder=3,
    )
    ax.add_feature(
        cfeature.COASTLINE.with_scale(COASTLINE_RES),
        linewidth=0.35, edgecolor="#444444", zorder=4,
    )
    ax.add_feature(
        cfeature.BORDERS.with_scale(COASTLINE_RES),
        linewidth=0.18, linestyle=":", edgecolor="#888888", zorder=4,
    )
    if gridlines:
        gl = ax.gridlines(
            draw_labels=True, linewidth=0.3,
            color="gray", alpha=0.5, zorder=5,
        )
        gl.top_labels = False
        gl.right_labels = False
        gl.xlabel_style = {"size": 9}
        gl.ylabel_style = {"size": 9}


def _get_coord(ds, options):
    for name in options:
        if name in ds.coords:
            return name
    raise KeyError(f"None of {options} found in {list(ds.coords)}")


def _discrete_cmap(vabs, cmap_name, n=13):
    boundaries = np.linspace(-vabs, vabs, n + 1)
    cmap = plt.get_cmap(cmap_name, n)
    norm = mpl.colors.BoundaryNorm(boundaries, ncolors=n)
    return cmap, boundaries, norm


def _green_pink_cmap(n=13):
    """Diverging green (negative) → white (zero) → pink (positive) colormap."""
    from matplotlib.colors import LinearSegmentedColormap
    colors = [
        "#1a7a4a",
        "#4dac58",
        "#80c97a",
        "#b3e2a0",
        "#d9f0c8",
        "#ffffff",
        "#fcd5e0",
        "#f9aabb",
        "#f47fa0",
        "#e04c7a",
        "#c0235a",
    ]
    cmap = LinearSegmentedColormap.from_list("green_pink", colors, N=n)
    return cmap


def _add_panel_label(ax, label, x=-0.06, y=1.04):
    ax.text(
        x, y, label,
        transform=ax.transAxes,
        fontsize=14, fontweight="bold",
        va="top", ha="left",
    )


def _horizontal_colorbar(fig, mappable, ax_ref, label,
                          pad=0.012, height=0.010, extend="both"):
    """Place a slim horizontal colorbar below ax_ref."""
    pos = ax_ref.get_position()
    cax = fig.add_axes([
        pos.x0,
        pos.y0 - pad - height,
        pos.width,
        height,
    ])
    cb = fig.colorbar(mappable, cax=cax, orientation="horizontal",
                      extend=extend)
    cb.set_label(label, fontsize=11, labelpad=4)
    cb.ax.tick_params(labelsize=10, rotation=30, length=3, width=0.8)
    return cb


def _cyclic_1d_field(lon, data):
    """Add a cyclic longitude column so Robinson maps do not show a seam."""
    data_cyc, lon_cyc = add_cyclic_point(data, coord=lon)
    return lon_cyc, data_cyc


# ═══════════════════════════════════════════════════════════════════════════════
# ROW 1 – mean & std maps
# ═══════════════════════════════════════════════════════════════════════════════

def _draw_row1(fig, ax_a, ax_b):
    ds_mean = xr.open_dataset(MEAN_NC)
    ds_std = xr.open_dataset(STD_NC)

    lat_name = _get_coord(ds_mean, ["lat", "latitude", "y"])
    lon_name = _get_coord(ds_mean, ["lon", "longitude", "x"])

    mean_data = ds_mean["approx_diff_mean"].squeeze()
    std_data = ds_std["approx_diff_std"].squeeze()

    lon = ds_mean[lon_name].values
    lat = ds_mean[lat_name].values
    lon_cyc, mean_cyc = _cyclic_1d_field(lon, mean_data.values)
    _, std_cyc = _cyclic_1d_field(lon, std_data.values)

    # panel a: mean
    vmax = float(np.nanpercentile(np.abs(mean_cyc), 99))
    norm_mean = mpl.colors.TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
    im_a = ax_a.pcolormesh(
        lon_cyc, lat,
        mean_cyc,
        transform=DATA_PROJ, cmap="RdBu_r", norm=norm_mean,
        shading="auto", zorder=1,
    )
    _add_map_features(ax_a, gridlines=False)
    _add_panel_label(ax_a, "a")
    _horizontal_colorbar(fig, im_a, ax_a, "Mean UTCI difference (\u00b0C)")

    # panel b: std
    finite_std = std_cyc[np.isfinite(std_cyc)]
    vmax_std = float(np.nanpercentile(finite_std, 99))
    norm_std = mpl.colors.Normalize(vmin=0, vmax=vmax_std)
    im_b = ax_b.pcolormesh(
        lon_cyc, lat,
        std_cyc,
        transform=DATA_PROJ, cmap="YlOrRd", norm=norm_std,
        shading="auto", zorder=1,
    )
    _add_map_features(ax_b, gridlines=False)
    _add_panel_label(ax_b, "b")
    _horizontal_colorbar(
        fig, im_b, ax_b,
        "Standard deviation of UTCI difference (\u00b0C)",
        extend="max",
    )

    ds_mean.close()
    ds_std.close()


# ═══════════════════════════════════════════════════════════════════════════════
# ROW 2 – stress-hours maps
# ═══════════════════════════════════════════════════════════════════════════════

HOURS_PANEL_CONFIG = [
    dict(var="very_cold", title="Very Strong Cold Stress Hours (< \u221227 \u00b0C)"),
    dict(var="very_strong_heat", title="Very Strong Heat Stress Hours (> 38 \u00b0C)"),
]


def _draw_row2(fig, ax_c, ax_d):
    ds = xr.open_dataset(HOURS_NC)
    lon = ds["lon"].values
    lat = ds["lat"].values
    lon_cyc, _ = _cyclic_1d_field(lon, ds[HOURS_PANEL_CONFIG[0]["var"]].values)
    lon2d, lat2d = np.meshgrid(lon_cyc, lat)

    for ax, cfg, label in zip([ax_c, ax_d], HOURS_PANEL_CONFIG, ["c", "d"]):
        data = ds[cfg["var"]].values
        _, data_cyc = _cyclic_1d_field(lon, data)
        finite = data_cyc[np.isfinite(data_cyc)]
        vabs = float(np.nanpercentile(np.abs(finite), 99))
        if not np.isfinite(vabs) or vabs == 0:
            vabs = 1.0

        n = 13
        cmap = _green_pink_cmap(n)
        boundaries = np.linspace(-vabs, vabs, n + 1)
        norm = mpl.colors.BoundaryNorm(boundaries, ncolors=n)

        ax.pcolormesh(
            lon2d, lat2d, data_cyc,
            transform=DATA_PROJ, cmap=cmap, norm=norm,
            shading="auto", zorder=1,
        )
        _add_map_features(ax)
        ax.set_title(cfg["title"], pad=3)
        _add_panel_label(ax, label)

        cb = _horizontal_colorbar(
            fig,
            mpl.cm.ScalarMappable(norm=norm, cmap=cmap),
            ax,
            "UTCI stress difference (hours)",
        )
        cb.set_ticks(boundaries[::2])
        cb.ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f"))

    ds.close()


# ═══════════════════════════════════════════════════════════════════════════════
# ROW 3 – KDE density distributions
# ═══════════════════════════════════════════════════════════════════════════════

DIST_PANEL_CONFIG = [
    dict(var="extreme_cold", title="Extreme Cold Stress (< \u221240 \u00b0C)", color="#2166ac"),
    dict(var="extreme_heat", title="Extreme Heat Stress (> 46 \u00b0C)", color="#b2182b"),
]

N_BINS_HIST = 80
KDE_POINTS = 500
CLIP_PCT = 0.9


def _clip_data(arr, pct=CLIP_PCT):
    lo = np.nanpercentile(arr, pct)
    hi = np.nanpercentile(arr, 100 - pct)
    return arr[(arr >= lo) & (arr <= hi)]


def _draw_row3(ax_e, ax_f):
    ds = xr.open_dataset(PCT_NC)

    all_data = {}
    for cfg in DIST_PANEL_CONFIG:
        raw = ds[cfg["var"]].values.ravel()
        data = _clip_data(raw[np.isfinite(raw)])
        all_data[cfg["var"]] = data

    valid = np.concatenate(list(all_data.values()))
    x_lo, x_hi = valid.min(), valid.max()

    for ax, cfg, label in zip([ax_e, ax_f], DIST_PANEL_CONFIG, ["e", "f"]):
        data = all_data[cfg["var"]]
        color = cfg["color"]

        counts, edges = np.histogram(data, bins=N_BINS_HIST, range=(x_lo, x_hi))
        bw = edges[1] - edges[0]
        ax.bar(edges[:-1], counts, width=bw, align="edge",
               color=color, alpha=0.35, linewidth=0)

        kde = gaussian_kde(data, bw_method="scott")
        x_kde = np.linspace(x_lo, x_hi, KDE_POINTS)
        y_kde = kde(x_kde) * data.size * bw
        ax.plot(x_kde, y_kde, color=color, linewidth=2.2)

        median = np.median(data)
        ax.axvline(median, color=color, linewidth=1.0,
                   linestyle=":", alpha=0.9, zorder=3)

        ylim_top = ax.get_ylim()[1]
        ax.text(
            median, ylim_top * 0.93,
            f" {median:.1f}%",
            fontsize=10, color=color,
            va="top", ha="left" if median >= 0 else "right",
        )

        print(
            f"  Panel {label} – {cfg['title']}\n"
            f"    n      = {data.size:,}\n"
            f"    mean   = {np.mean(data):.2f}%\n"
            f"    median = {median:.2f}%\n"
            f"    std    = {np.std(data):.2f}%\n"
            f"    min    = {data.min():.2f}%\n"
            f"    max    = {data.max():.2f}%\n"
        )

        _add_panel_label(ax, label, x=-0.09, y=1.06)
        ax.set_title(cfg["title"], pad=3)
        ax.set_xlabel("UTCI stress difference (%)", fontsize=12, labelpad=4)
        ax.set_ylabel("Number of grid cells", fontsize=12, labelpad=4)
        ax.yaxis.set_major_formatter(
            mticker.FuncFormatter(lambda x, _: f"{int(x):,}")
        )
        ax.yaxis.grid(True, linewidth=0.3, alpha=0.4, linestyle="--")
        ax.set_axisbelow(True)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.tick_params(axis="both", length=4, width=0.8)

    ds.close()


# ═══════════════════════════════════════════════════════════════════════════════
# ASSEMBLE FIGURE
# ═══════════════════════════════════════════════════════════════════════════════

def build_figure():
    fig = plt.figure(figsize=(12, 11.5))

    gs = mpl.gridspec.GridSpec(
        3, 2,
        figure=fig,
        height_ratios=[1.0, 1.0, 0.75],
        hspace=0.45,   # smaller = less vertical space
        wspace=0.010,   # smaller = less horizontal space
        left=0.05, right=0.98,
        top=0.97, bottom=0.05,
    )

    ax_a = fig.add_subplot(gs[0, 0], projection=MAP_PROJ)
    ax_b = fig.add_subplot(gs[0, 1], projection=MAP_PROJ)
    ax_c = fig.add_subplot(gs[1, 0], projection=MAP_PROJ)
    ax_d = fig.add_subplot(gs[1, 1], projection=MAP_PROJ)
    ax_e = fig.add_subplot(gs[2, 0])
    ax_f = fig.add_subplot(gs[2, 1])

    print("Drawing row 1 (mean & std) …")
    _draw_row1(fig, ax_a, ax_b)

    print("Drawing row 2 (stress hours) …")
    _draw_row2(fig, ax_c, ax_d)

    print("Drawing row 3 (distributions) …")
    _draw_row3(ax_e, ax_f)

    for ext in [".png", ".pdf", ".eps"]:
        out = OUTPUT_PNG.replace(".png", ext)
        fig.savefig(out, dpi=600, bbox_inches="tight", pad_inches=0.01,
                    facecolor="white")
        print(f"  ✓  Saved → {out}")
    print()
    plt.show()


if __name__ == "__main__":
    build_figure()

In [ ]:
"""
utci_six_panel_figure.py
========================
Publication-ready 6-panel figure (3 rows × 2 columns) for Nature submission.

  Row 1 (a, b) : Annual mean & std of UTCI approximation difference
  Row 2 (c, d) : Yearly stress-hours difference – Very Strong Cold / Very Strong Heat
  Row 3 (e, f) : KDE density distributions – Extreme Cold / Extreme Heat % change

Adjust the three DATA PATHS blocks below to match your file locations,
then run:  python utci_six_panel_figure.py
"""

import os
import numpy as np
import xarray as xr
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from scipy.stats import gaussian_kde

# ═══════════════════════════════════════════════════════════════════════════════
# DATA PATHS  ── edit these three blocks ────────────────────────────────────────
# ═══════════════════════════════════════════════════════════════════════════════

# Row 1 – mean / std maps
MEAN_STD_DIR = (
    r"C:\Users\nerc-user\OneDrive - Nexus365"
    r"\UTCI_NN\ERA5_exploration\data\utci_diff_mean_var"
)
MEAN_NC = os.path.join(MEAN_STD_DIR, "utci_diff_2024_mean.nc")
STD_NC  = os.path.join(MEAN_STD_DIR, "utci_diff_2024_std.nc")

# Row 2 – stress-hours maps
HOURS_NC = "utci_yearly_land_hours.nc"

# Row 3 – % change distributions
PCT_NC = "utci_yearly_land_pct_change.nc"

# Output
OUTPUT_PNG = "utci_six_panel_nature.png"

# ═══════════════════════════════════════════════════════════════════════════════
# GLOBAL STYLE  (Nature guidelines: 7 pt minimum, Helvetica/Arial)
# ═══════════════════════════════════════════════════════════════════════════════

mpl.rcParams.update({
    # ── matched to companion figure style ────────────────────────────────────
    "figure.dpi":        300,
    "savefig.dpi":       600,
    "font.family":       "serif",
    "font.serif":        ["Times New Roman", "Times", "DejaVu Serif"],
    "font.size":         12,
    "axes.titlesize":    13,
    "axes.labelsize":    12,
    "axes.linewidth":    1.0,
    "xtick.labelsize":   11,
    "ytick.labelsize":   11,
    "legend.fontsize":   11,
    "legend.frameon":    False,
    "lines.linewidth":   1.5,
    "lines.markersize":  6,
    "axes.grid":         False,
    "pdf.fonttype":      42,
    "ps.fonttype":       42,
    "xtick.major.width": 1.0,
    "ytick.major.width": 1.0,
    "patch.linewidth":   1.0,
})

PANEL_LABELS = list("abcdef")

# ═══════════════════════════════════════════════════════════════════════════════
# SHARED HELPERS
# ═══════════════════════════════════════════════════════════════════════════════

MAP_EXTENT   = [-180, 180, -60, 90]
COASTLINE_RES = "110m"


def _add_map_features(ax, gridlines=False):
    """
    Map background for Nature figures.
    Ocean painted ON TOP of data (zorder=3) so ocean grid cells are masked white.
    Land underlay gives a light-grey base for land cells without data.
    """
    ax.add_feature(
        cfeature.LAND.with_scale(COASTLINE_RES),
        facecolor="#f4f4f4", zorder=0,
    )
    ax.add_feature(
        cfeature.OCEAN.with_scale(COASTLINE_RES),
        facecolor="white", zorder=3,    # on top of pcolormesh (zorder=1)
    )
    ax.add_feature(
        cfeature.COASTLINE.with_scale(COASTLINE_RES),
        linewidth=0.35, edgecolor="#444444", zorder=4,
    )
    ax.add_feature(
        cfeature.BORDERS.with_scale(COASTLINE_RES),
        linewidth=0.18, linestyle=":", edgecolor="#888888", zorder=4,
    )
    if gridlines:
        gl = ax.gridlines(
            draw_labels=True, linewidth=0.3,
            color="gray", alpha=0.5, zorder=5,
        )
        gl.top_labels   = False
        gl.right_labels = False
        gl.xlabel_style = {"size": 9}
        gl.ylabel_style = {"size": 9}
    ax.set_extent(MAP_EXTENT, crs=ccrs.PlateCarree())


def _get_coord(ds, options):
    for name in options:
        if name in ds.coords:
            return name
    raise KeyError(f"None of {options} found in {list(ds.coords)}")


def _discrete_cmap(vabs, cmap_name, n=13):
    boundaries = np.linspace(-vabs, vabs, n + 1)
    cmap = plt.get_cmap(cmap_name, n)
    norm = mpl.colors.BoundaryNorm(boundaries, ncolors=n)
    return cmap, boundaries, norm


def _green_pink_cmap(n=13):
    """Diverging green (negative) → white (zero) → pink (positive) colormap."""
    from matplotlib.colors import LinearSegmentedColormap
    colors = [
        "#1a7a4a",   # deep green  (most negative)
        "#4dac58",
        "#80c97a",
        "#b3e2a0",
        "#d9f0c8",
        "#ffffff",   # white (zero)
        "#fcd5e0",
        "#f9aabb",
        "#f47fa0",
        "#e04c7a",
        "#c0235a",   # deep pink   (most positive)
    ]
    cmap = LinearSegmentedColormap.from_list("green_pink", colors, N=n)
    return cmap


def _add_panel_label(ax, label, x=-0.06, y=1.04):
    ax.text(
        x, y, label,
        transform=ax.transAxes,
        fontsize=14, fontweight="bold",
        va="top", ha="left",
    )


def _horizontal_colorbar(fig, mappable, ax_ref, label,
                          pad=0.012, height=0.010, extend="both"):
    """Place a slim horizontal colorbar below ax_ref."""
    pos = ax_ref.get_position()
    cax = fig.add_axes([
        pos.x0,
        pos.y0 - pad - height,
        pos.width,
        height,
    ])
    cb = fig.colorbar(mappable, cax=cax, orientation="horizontal",
                      extend=extend)
    cb.set_label(label, fontsize=11, labelpad=4)
    cb.ax.tick_params(labelsize=10, rotation=30, length=3, width=0.8)
    return cb


# ═══════════════════════════════════════════════════════════════════════════════
# ROW 1 – mean & std maps
# ═══════════════════════════════════════════════════════════════════════════════

def _draw_row1(fig, ax_a, ax_b):
    ds_mean = xr.open_dataset(MEAN_NC)
    ds_std  = xr.open_dataset(STD_NC)

    lat_name = _get_coord(ds_mean, ["lat", "latitude", "y"])
    lon_name = _get_coord(ds_mean, ["lon", "longitude", "x"])

    mean_data = ds_mean["approx_diff_mean"].squeeze()
    std_data  = ds_std["approx_diff_std"].squeeze()

    proj = ccrs.PlateCarree()

    # ── panel a: mean ──────────────────────────────────────────────────────────
    vmax = float(np.nanpercentile(np.abs(mean_data.values), 99))
    norm_mean = mpl.colors.TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
    im_a = ax_a.pcolormesh(
        ds_mean[lon_name].values, ds_mean[lat_name].values,
        mean_data.values,
        transform=proj, cmap="RdBu_r", norm=norm_mean,
        shading="auto", zorder=1,
    )
    _add_map_features(ax_a, gridlines=False)
    #ax_a.set_title("2024 mean UTCI difference (NN − Polynomial)", pad=5)
    _add_panel_label(ax_a, "a")
    _horizontal_colorbar(fig, im_a, ax_a, "Mean UTCI difference (°C)")

    # ── panel b: std ───────────────────────────────────────────────────────────
    vmax_std = float(np.nanpercentile(std_data.values[np.isfinite(std_data.values)], 99))
    norm_std = mpl.colors.Normalize(vmin=0, vmax=vmax_std)
    im_b = ax_b.pcolormesh(
        ds_std[lon_name].values, ds_std[lat_name].values,
        std_data.values,
        transform=proj, cmap="YlOrRd", norm=norm_std,
        shading="auto", zorder=1,
    )
    _add_map_features(ax_b, gridlines=False)
    #ax_b.set_title("Annual std of UTCI difference (NN − Polynomial)", pad=5)
    _add_panel_label(ax_b, "b")
    _horizontal_colorbar(fig, im_b, ax_b, "Standard deviation of UTCI difference (°C)", extend="max")

    ds_mean.close(); ds_std.close()


# ═══════════════════════════════════════════════════════════════════════════════
# ROW 2 – stress-hours maps
# ═══════════════════════════════════════════════════════════════════════════════

HOURS_PANEL_CONFIG = [
    dict(var="very_cold",        title="Very Strong Cold Stress Hours (< \u221227 \u00b0C)"),
    dict(var="very_strong_heat", title="Very Strong Heat Stress Hours (> 38 \u00b0C)"),
]


def _draw_row2(fig, ax_c, ax_d):
    ds   = xr.open_dataset(HOURS_NC)
    lon  = ds["lon"].values
    lat  = ds["lat"].values
    lon2d, lat2d = np.meshgrid(lon, lat)
    proj = ccrs.PlateCarree()

    for ax, cfg, label in zip([ax_c, ax_d], HOURS_PANEL_CONFIG, ["c", "d"]):
        data = ds[cfg["var"]].values
        vabs = float(np.nanpercentile(np.abs(data[np.isfinite(data)]), 99))
        if not np.isfinite(vabs) or vabs == 0:
            vabs = 1.0

        n          = 13
        cmap       = _green_pink_cmap(n)
        boundaries = np.linspace(-vabs, vabs, n + 1)
        norm       = mpl.colors.BoundaryNorm(boundaries, ncolors=n)

        ax.pcolormesh(
            lon2d, lat2d, data,
            transform=proj, cmap=cmap, norm=norm,
            shading="auto", zorder=1,
        )
        _add_map_features(ax)
        ax.set_title(cfg["title"], pad=3)
        _add_panel_label(ax, label)

        cb = _horizontal_colorbar(
            fig,
            mpl.cm.ScalarMappable(norm=norm, cmap=cmap),
            ax,
            "UTCI stress-hour difference (hours)",
        )
        cb.set_ticks(boundaries[::2])
        cb.ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f"))

    ds.close()


# ═══════════════════════════════════════════════════════════════════════════════
# ROW 3 – KDE density distributions
# ═══════════════════════════════════════════════════════════════════════════════

DIST_PANEL_CONFIG = [
    dict(var="extreme_cold", title="Extreme Cold Stress (< −40 °C)", color="#2166ac"),
    dict(var="extreme_heat", title="Extreme Heat Stress (> 46 °C)",  color="#b2182b"),
]

N_BINS_HIST  = 80
KDE_POINTS   = 500
CLIP_PCT     = 0.9


def _clip_data(arr, pct=CLIP_PCT):
    lo = np.nanpercentile(arr, pct)
    hi = np.nanpercentile(arr, 100 - pct)
    return arr[(arr >= lo) & (arr <= hi)]


def _draw_row3(ax_e, ax_f):
    ds = xr.open_dataset(PCT_NC)

    all_data = {}
    for cfg in DIST_PANEL_CONFIG:
        raw  = ds[cfg["var"]].values.ravel()
        data = _clip_data(raw[np.isfinite(raw)])
        all_data[cfg["var"]] = data

    valid       = np.concatenate(list(all_data.values()))
    x_lo, x_hi = valid.min(), valid.max()

    for ax, cfg, label in zip([ax_e, ax_f], DIST_PANEL_CONFIG, ["e", "f"]):
        data  = all_data[cfg["var"]]
        color = cfg["color"]

        # Histogram bars
        counts, edges = np.histogram(data, bins=N_BINS_HIST, range=(x_lo, x_hi))
        bw = edges[1] - edges[0]
        ax.bar(edges[:-1], counts, width=bw, align="edge",
               color=color, alpha=0.35, linewidth=0)

        # KDE curve scaled to counts
        kde   = gaussian_kde(data, bw_method="scott")
        x_kde = np.linspace(x_lo, x_hi, KDE_POINTS)
        y_kde = kde(x_kde) * data.size * bw
        ax.plot(x_kde, y_kde, color=color, linewidth=2.2)

        # Reference lines
        # ax.axvline(0,      color="black", linewidth=0.7,
        #            linestyle="--", alpha=0.55, zorder=3)
        median = np.median(data)
        ax.axvline(median, color=color,   linewidth=1.0,
                   linestyle=":",  alpha=0.9,  zorder=3)

        # Median label on plot
        ylim_top = ax.get_ylim()[1]
        ax.text(
            median, ylim_top * 0.93,
            f" {median:.1f}%",
            fontsize=10, color=color,
            va="top", ha="left" if median >= 0 else "right",
        )

        # Print stats to console (no box on figure)
        print(
            f"  Panel {label} – {cfg['title']}\n"
            f"    n      = {data.size:,}\n"
            f"    mean   = {np.mean(data):.2f}%\n"
            f"    median = {median:.2f}%\n"
            f"    std    = {np.std(data):.2f}%\n"
            f"    min    = {data.min():.2f}%\n"
            f"    max    = {data.max():.2f}%\n"
        )

        _add_panel_label(ax, label, x=-0.09, y=1.06)
        ax.set_title(cfg["title"], pad=3)
        ax.set_xlabel("UTCI stress-hour difference (%)", fontsize=12, labelpad=4)
        ax.set_ylabel("Number of grid cells", fontsize=12, labelpad=4)
        ax.yaxis.set_major_formatter(
            mticker.FuncFormatter(lambda x, _: f"{int(x):,}")
        )
        ax.yaxis.grid(True, linewidth=0.3, alpha=0.4, linestyle="--")
        ax.set_axisbelow(True)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.tick_params(axis="both", length=4, width=0.8)

    ds.close()


# ═══════════════════════════════════════════════════════════════════════════════
# ASSEMBLE FIGURE
# ═══════════════════════════════════════════════════════════════════════════════

def build_figure():
    proj = ccrs.PlateCarree()

    # Nature single-column = 89 mm wide; double-column = 183 mm wide.
    # At 300 dpi, 183 mm ≈ 7.2 inches.  Height chosen for 3 rows with maps.
    fig = plt.figure(figsize=(10, 14))

    # ── GridSpec: 3 rows, 2 cols ─────────────────────────────────────────────
    # Rows 1 & 2 are maps (taller), row 3 is histogram (shorter).
    # Extra vertical space at the bottom of each map row is used for colourbars.
    gs = mpl.gridspec.GridSpec(
        3, 2,
        figure=fig,
        height_ratios=[1.0, 1.0, 0.75],
        hspace=0.55,   # vertical gap between rows (room for colourbars + titles)
        wspace=0.18,   # match companion figure wspace
        left=0.06, right=0.97,
        top=0.96, bottom=0.06,
    )

    ax_a = fig.add_subplot(gs[0, 0], projection=proj)
    ax_b = fig.add_subplot(gs[0, 1], projection=proj)
    ax_c = fig.add_subplot(gs[1, 0], projection=proj)
    ax_d = fig.add_subplot(gs[1, 1], projection=proj)
    ax_e = fig.add_subplot(gs[2, 0])
    ax_f = fig.add_subplot(gs[2, 1])

    # Fix map aspect ratio for all cartopy axes
    # for ax in [ax_a, ax_b, ax_c, ax_d]:
    #     ax.set_aspect("auto")

    print("Drawing row 1 (mean & std) …")
    _draw_row1(fig, ax_a, ax_b)

    print("Drawing row 2 (stress hours) …")
    _draw_row2(fig, ax_c, ax_d)

    print("Drawing row 3 (distributions) …")
    _draw_row3(ax_e, ax_f)

    for ext in [".png", ".pdf", ".eps"]:
        out = OUTPUT_PNG.replace(".png", ext)
        fig.savefig(out, dpi=600, bbox_inches="tight", pad_inches=0.01,
                    facecolor="white")
        print(f"  ✓  Saved → {out}")
    print()
    plt.show()


# ═══════════════════════════════════════════════════════════════════════════════
if __name__ == "__main__":
    build_figure()

In [ ]:
import os
import numpy as np
import xarray as xr
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from scipy.stats import gaussian_kde

# ═══════════════════════════════════════════════════════════════════════════════
# DATA PATHS
# ═══════════════════════════════════════════════════════════════════════════════

MEAN_STD_DIR = (
    r"C:\Users\nerc-user\OneDrive - Nexus365"
    r"\UTCI_NN\ERA5_exploration\data\utci_diff_mean_var"
)
MEAN_NC = os.path.join(MEAN_STD_DIR, "utci_diff_2024_mean.nc")
STD_NC  = os.path.join(MEAN_STD_DIR, "utci_diff_2024_std.nc")

HOURS_NC = "utci_yearly_land_hours.nc"
PCT_NC   = "utci_yearly_land_pct_change.nc"

OUTPUT_PNG = "utci_six_panel_nature.png"

# ═══════════════════════════════════════════════════════════════════════════════
# STYLE
# ═══════════════════════════════════════════════════════════════════════════════

mpl.rcParams.update({
    "figure.dpi": 300,
    "savefig.dpi": 600,
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "font.size": 12,
})

# ═══════════════════════════════════════════════════════════════════════════════
# PROJECTION
# ═══════════════════════════════════════════════════════════════════════════════

MAP_PROJ = ccrs.Robinson()
DATA_CRS = ccrs.PlateCarree()

# ═══════════════════════════════════════════════════════════════════════════════
# CACHE FEATURES (speed boost)
# ═══════════════════════════════════════════════════════════════════════════════

LAND = cfeature.LAND.with_scale("110m")
OCEAN = cfeature.OCEAN.with_scale("110m")
COAST = cfeature.COASTLINE.with_scale("110m")
BORDERS = cfeature.BORDERS.with_scale("110m")

# ═══════════════════════════════════════════════════════════════════════════════
# HELPERS
# ═══════════════════════════════════════════════════════════════════════════════

def _add_map_features(ax):
    ax.add_feature(LAND, facecolor="#f4f4f4", zorder=0)
    ax.add_feature(OCEAN, facecolor="white", zorder=3)
    ax.add_feature(COAST, linewidth=0.35, edgecolor="#444444", zorder=4)
    ax.add_feature(BORDERS, linewidth=0.18, linestyle=":", edgecolor="#888888", zorder=4)


def _add_panel_label(ax, label, x=-0.06, y=1.04):
    ax.text(x, y, label, transform=ax.transAxes,
            fontsize=14, fontweight="bold", va="top")


def _horizontal_colorbar(fig, mappable, ax_ref, label,
                        pad=0.008, height=0.008, extend="both"):
    pos = ax_ref.get_position()
    cax = fig.add_axes([
        pos.x0,
        pos.y0 - pad - height,
        pos.width,
        height,
    ])
    cb = fig.colorbar(mappable, cax=cax,
                      orientation="horizontal", extend=extend)
    cb.set_label(label)
    return cb


def _coarsen(da, factor=2):
    """Downsample data for faster plotting"""
    return da.coarsen(lat=factor, lon=factor, boundary="trim").mean()


# ═══════════════════════════════════════════════════════════════════════════════
# ROW 1
# ═══════════════════════════════════════════════════════════════════════════════

def _draw_row1(fig, ax_a, ax_b):
    ds_mean = xr.open_dataset(MEAN_NC)
    ds_std  = xr.open_dataset(STD_NC)

    mean_data = _coarsen(ds_mean["approx_diff_mean"].squeeze(), 2)
    std_data  = _coarsen(ds_std["approx_diff_std"].squeeze(), 2)

    vmax = float(np.nanpercentile(np.abs(mean_data.values), 99))

    im_a = ax_a.pcolormesh(
        mean_data.lon, mean_data.lat, mean_data,
        transform=DATA_CRS,
        cmap="RdBu_r",
        vmin=-vmax, vmax=vmax,
        shading="auto",
        rasterized=True
    )
    _add_map_features(ax_a)
    _add_panel_label(ax_a, "a")
    _horizontal_colorbar(fig, im_a, ax_a, "Mean UTCI difference (°C)")

    vmax_std = float(np.nanpercentile(std_data.values, 99))

    im_b = ax_b.pcolormesh(
        std_data.lon, std_data.lat, std_data,
        transform=DATA_CRS,
        cmap="YlOrRd",
        vmin=0, vmax=vmax_std,
        shading="auto",
        rasterized=True
    )
    _add_map_features(ax_b)
    _add_panel_label(ax_b, "b")
    _horizontal_colorbar(fig, im_b, ax_b,
                         "Standard deviation (°C)", extend="max")

    ds_mean.close(); ds_std.close()


# ═══════════════════════════════════════════════════════════════════════════════
# ROW 2
# ═══════════════════════════════════════════════════════════════════════════════

def _draw_row2(fig, ax_c, ax_d):
    ds = xr.open_dataset(HOURS_NC)

    lon, lat = ds["lon"], ds["lat"]
    lon2d, lat2d = np.meshgrid(lon, lat)

    for ax, var, label in zip(
        [ax_c, ax_d],
        ["very_cold", "very_strong_heat"],
        ["c", "d"]
    ):
        data = ds[var].values

        # Downsample manually
        data = xr.DataArray(data, dims=("lat", "lon"),
                            coords={"lat": lat, "lon": lon})
        data = _coarsen(data, 2)

        vmax = float(np.nanpercentile(np.abs(data.values), 99))

        im = ax.pcolormesh(
            data.lon, data.lat, data,
            transform=DATA_CRS,
            cmap="RdBu_r",
            vmin=-vmax, vmax=vmax,
            shading="auto",
            rasterized=True
        )

        _add_map_features(ax)
        _add_panel_label(ax, label)
        _horizontal_colorbar(fig, im, ax,
                             "UTCI stress-hour difference (hours)")

    ds.close()


# ═══════════════════════════════════════════════════════════════════════════════
# ROW 3
# ═══════════════════════════════════════════════════════════════════════════════

def _draw_row3(ax_e, ax_f):
    ds = xr.open_dataset(PCT_NC)

    all_data = {}
    for var in ["extreme_cold", "extreme_heat"]:
        data = ds[var].values.ravel()
        data = data[np.isfinite(data)]

        # clip extremes (stabilises KDE)
        lo = np.nanpercentile(data, 0.5)
        hi = np.nanpercentile(data, 99.5)
        data = data[(data >= lo) & (data <= hi)]

        all_data[var] = data

    # shared x-range
    combined = np.concatenate(list(all_data.values()))
    x_min, x_max = combined.min(), combined.max()

    for ax, var, label, color in zip(
        [ax_e, ax_f],
        ["extreme_cold", "extreme_heat"],
        ["e", "f"],
        ["#2166ac", "#b2182b"]
    ):
        data = all_data[var]

        # histogram
        counts, bins = np.histogram(data, bins=60, range=(x_min, x_max))
        width = bins[1] - bins[0]

        ax.bar(
            bins[:-1], counts,
            width=width,
            align="edge",
            color=color,
            alpha=0.35,
            linewidth=0
        )

        # KDE scaled to counts
        kde = gaussian_kde(data)
        x = np.linspace(x_min, x_max, 400)
        y = kde(x) * len(data) * width

        ax.plot(x, y, color=color, linewidth=2)

        # median line
        median = np.median(data)
        ax.axvline(median, color=color, linestyle=":", linewidth=1)

        # formatting
        ax.set_xlim(x_min, x_max)
        ax.set_xlabel("UTCI stress-hour difference (%)")
        ax.set_ylabel("Grid cell count")

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        ax.grid(True, linestyle="--", linewidth=0.3, alpha=0.4)
        ax.set_axisbelow(True)

        _add_panel_label(ax, label, x=-0.09, y=1.05)

    ds.close()

# ═══════════════════════════════════════════════════════════════════════════════
# BUILD FIGURE
# ═══════════════════════════════════════════════════════════════════════════════

def build_figure():
    fig = plt.figure(figsize=(10, 14))

    gs = mpl.gridspec.GridSpec(
        3, 2,
        height_ratios=[1.1, 1.1, 0.65],  # give maps more space
        hspace=0.32,   # ↓ tighter vertically
        wspace=0.015,  # ↓ almost touching
        left=0.035,
        right=0.99,
        top=0.975,
        bottom=0.05,
    )

    ax_a = fig.add_subplot(gs[0, 0], projection=MAP_PROJ)
    ax_b = fig.add_subplot(gs[0, 1], projection=MAP_PROJ)
    ax_c = fig.add_subplot(gs[1, 0], projection=MAP_PROJ)
    ax_d = fig.add_subplot(gs[1, 1], projection=MAP_PROJ)
    ax_e = fig.add_subplot(gs[2, 0])
    ax_f = fig.add_subplot(gs[2, 1])

    # keep maps centred and tight
    for ax in [ax_a, ax_b, ax_c, ax_d]:
        ax.set_anchor("C")

    print("Row 1...")
    _draw_row1(fig, ax_a, ax_b)

    print("Row 2...")
    _draw_row2(fig, ax_c, ax_d)

    print("Row 3...")
    _draw_row3(ax_e, ax_f)

    for ext in [".png", ".pdf"]:
        fig.savefig(OUTPUT_PNG.replace(".png", ext),
                    dpi=600, bbox_inches="tight")

    print("Done.")


# ═══════════════════════════════════════════════════════════════════════════════
if __name__ == "__main__":
    build_figure()

## Elevation

In [ ]:
"""
utci_elevation_correlation.py
==============================
Correlation between land-surface elevation and the annual mean UTCI
approximation difference (NN − Polynomial).

Elevation source — cartopy NaturalEarth raster (zero extra installs)
---------------------------------------------------------------------
Cartopy automatically downloads and caches the NaturalEarth
Cross-Blended Hypsometric Tints + Shaded Relief GeoTIFF (~30 MB) the
first time this script runs.  No account, no API key, no pip install
beyond what the six-panel figure already requires.

The raster is a visual product (not a true DEM) so pixel luminance is
used as an elevation proxy: bright = high terrain, dark = low/ocean.
At ERA5 resolution (~0.25°) this gives a reliable elevation signal for
correlation analysis.

Steps
-----
1. Load mean UTCI difference      (utci_diff_2024_mean.nc)
2. Download/cache NE raster via cartopy, read with Pillow
3. Interpolate raster luminance to UTCI grid
4. Apply land mask (cartopy 110 m shapefile — already used in six-panel fig)
5. Pearson r, Spearman ρ, OLS regression → print to console
6. Scatter plot coloured by |latitude|, OLS line, stats annotated
7. Save PNG + PDF + EPS at 600 dpi

Only edit the CONFIG block, then run:
    python utci_elevation_correlation.py
"""

import os
import io
import zipfile
import urllib.request
import numpy as np
import xarray as xr
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy import stats

import cartopy
from cartopy.io import shapereader
from shapely.geometry import Point
from shapely.ops import unary_union
from shapely.prepared import prep

# ═══════════════════════════════════════════════════════════════════════════════
# CONFIG  ── only edit this block ───────────────────────────────────────────────
# ═══════════════════════════════════════════════════════════════════════════════

MEAN_STD_DIR = (
    r"C:\Users\nerc-user\OneDrive - Nexus365"
    r"\UTCI_NN\ERA5_exploration\data\utci_diff_mean_var"
)
MEAN_NC     = os.path.join(MEAN_STD_DIR, "utci_diff_2024_mean.nc")
OUTPUT_BASE = "utci_elevation_correlation"   # .png / .pdf / .eps appended

# ═══════════════════════════════════════════════════════════════════════════════
# STYLE  ── consistent with companion figures ───────────────────────────────────
# ═══════════════════════════════════════════════════════════════════════════════

mpl.rcParams.update({
    "figure.dpi":        300,
    "savefig.dpi":       600,
    "font.family":       "serif",
    "font.serif":        ["Times New Roman", "Times", "DejaVu Serif"],
    "font.size":         12,
    "axes.titlesize":    13,
    "axes.labelsize":    12,
    "axes.linewidth":    1.0,
    "xtick.labelsize":   11,
    "ytick.labelsize":   11,
    "legend.fontsize":   11,
    "legend.frameon":    False,
    "lines.linewidth":   1.5,
    "axes.grid":         False,
    "pdf.fonttype":      42,
    "ps.fonttype":       42,
    "xtick.major.width": 1.0,
    "ytick.major.width": 1.0,
})

# ═══════════════════════════════════════════════════════════════════════════════
# HELPERS
# ═══════════════════════════════════════════════════════════════════════════════

def _get_coord(ds, options):
    for name in options:
        if name in ds.coords:
            return name
    raise KeyError(f"None of {options} found in {list(ds.coords)}")


# ═══════════════════════════════════════════════════════════════════════════════
# STEP 1 – load UTCI mean difference
# ═══════════════════════════════════════════════════════════════════════════════

def load_utci():
    print("Loading UTCI mean difference …")
    ds       = xr.open_dataset(MEAN_NC)
    lat_name = _get_coord(ds, ["lat", "latitude", "y"])
    lon_name = _get_coord(ds, ["lon", "longitude", "x"])
    diff     = ds["approx_diff_mean"].squeeze().values
    lats     = ds[lat_name].values
    lons     = ds[lon_name].values
    ds.close()
    print(f"  Grid: {len(lats)} lats × {len(lons)} lons")
    return diff, lats, lons


# ═══════════════════════════════════════════════════════════════════════════════
# STEP 2 – get NaturalEarth raster path (download once via cartopy cache)
# ═══════════════════════════════════════════════════════════════════════════════

def _get_ne_raster_path():
    """
    Return local path to the NaturalEarth shaded-relief GeoTIFF,
    downloading it into cartopy's cache directory if not already present.
    """
    cache_dir = os.path.join(
        cartopy.config["data_dir"], "ne_raster"
    )
    os.makedirs(cache_dir, exist_ok=True)

    # Check if already cached
    for fname in ["NE1_HR_LC_SR_W_DR.tif",
                  "NE1_HR_LC_SR_W_DR.tiff",
                  "NE1_HR_LC_SR_W_DR.jpg"]:
        candidate = os.path.join(cache_dir, fname)
        if os.path.exists(candidate):
            print(f"  Using cached raster: {candidate}")
            return candidate

    # Download the zip and extract
    url = ("https://naturalearth.s3.amazonaws.com/raster/"
           "NE1_HR_LC_SR_W_DR.zip")
    print(f"  Downloading NaturalEarth raster from {url}")
    print("  (This ~30 MB download only happens once; cached locally afterwards)")
    with urllib.request.urlopen(url) as resp:
        raw = resp.read()

    with zipfile.ZipFile(io.BytesIO(raw)) as zf:
        for member in zf.namelist():
            if member.lower().endswith((".tif", ".tiff", ".jpg")):
                zf.extract(member, cache_dir)
                out = os.path.join(cache_dir, member)
                print(f"  Saved → {out}")
                return out

    raise FileNotFoundError(
        "Could not locate an image file inside the NaturalEarth zip.\n"
        "Download NE1_HR_LC_SR_W_DR.zip manually from "
        "https://www.naturalearthdata.com/downloads/10m-raster-data/"
        "10m-natural-earth-1/ and extract the .tif alongside this script."
    )


# ═══════════════════════════════════════════════════════════════════════════════
# STEP 3 – build elevation proxy grid
# ═══════════════════════════════════════════════════════════════════════════════

def build_elevation_grid(lats, lons):
    """
    Read the NaturalEarth raster and interpolate luminance (elevation proxy)
    onto the UTCI lat/lon grid.

    Requires Pillow — installed alongside matplotlib in most scientific
    Python environments.  If missing: pip install Pillow
    """
    try:
        from PIL import Image
    except ImportError:
        raise ImportError(
            "Pillow is required to read the NaturalEarth raster.\n"
            "Install it with:  pip install Pillow"
        )

    img_path = _get_ne_raster_path()
    print("  Reading raster and computing luminance …")
    img     = Image.open(img_path).convert("RGB")
    arr     = np.array(img, dtype=np.float32)          # (H, W, 3)
    # ITU-R BT.601 luminance: bright pixels = high terrain
    lum     = 0.299 * arr[:, :, 0] + 0.587 * arr[:, :, 1] + 0.114 * arr[:, :, 2]

    H, W    = lum.shape
    # NaturalEarth global raster: lon −180…180 left→right, lat +90…−90 top→bottom
    img_lons = np.linspace(-180.0, 180.0, W, endpoint=False) + 180.0 / W
    img_lats = np.linspace( 90.0, -90.0, H, endpoint=False) -  90.0 / H

    print("  Interpolating raster to UTCI grid …")
    da_lum = xr.DataArray(
        lum,
        dims=("ilat", "ilon"),
        coords={"ilat": img_lats, "ilon": img_lons},
    )
    # Linear interpolation — fast with xarray on a regular grid
    elev_grid = da_lum.interp(ilat=lats, ilon=lons, method="linear").values
    print(f"  Luminance range on UTCI grid: "
          f"{np.nanmin(elev_grid):.1f} – {np.nanmax(elev_grid):.1f}")
    return elev_grid


# ═══════════════════════════════════════════════════════════════════════════════
# STEP 4 – land mask (cartopy 110 m shapefile, same as six-panel figure)
# ═══════════════════════════════════════════════════════════════════════════════

def build_land_mask(lats, lons):
    print("Building land mask (cartopy 110 m shapefile) …")
    shp   = shapereader.natural_earth("110m", "physical", "land")
    geoms = [g.geometry for g in shapereader.Reader(shp).records()]
    land  = prep(unary_union(geoms))

    lon2d, lat2d = np.meshgrid(lons, lats)
    flat  = lon2d.size
    mask  = np.empty(flat, dtype=bool)
    chunk = 500_000
    for i in range(0, flat, chunk):
        pts = [Point(xy) for xy in zip(lon2d.ravel()[i:i+chunk],
                                       lat2d.ravel()[i:i+chunk])]
        mask[i:i+chunk] = [land.contains(p) for p in pts]

    print(f"  Land pixels: {mask.sum():,} / {flat:,}")
    return mask.reshape(lat2d.shape), lat2d


# ═══════════════════════════════════════════════════════════════════════════════
# STEP 5 – statistics
# ═══════════════════════════════════════════════════════════════════════════════

def compute_stats(elev, diff):
    pearson_r,  pearson_p  = stats.pearsonr(elev, diff)
    spearman_r, spearman_p = stats.spearmanr(elev, diff)
    slope, intercept, *_   = stats.linregress(elev, diff)

    print("\n── Correlation statistics ──────────────────────────────────────────────")
    print(f"  N (land pixels)  : {len(elev):,}")
    print(f"  Pearson  r       : {pearson_r:+.4f}   (p = {pearson_p:.2e})")
    print(f"  Spearman ρ       : {spearman_r:+.4f}   (p = {spearman_p:.2e})")
    print(f"  OLS slope        : {slope:+.4e}  °C per luminance unit")
    print(f"  OLS intercept    : {intercept:+.4f} °C")
    print(f"  Note: elevation proxy = NaturalEarth luminance")
    print(f"        (0 = dark / low terrain, 255 = bright / high terrain)")
    print("────────────────────────────────────────────────────────────────────────\n")

    return pearson_r, pearson_p, spearman_r, spearman_p, slope, intercept


# ═══════════════════════════════════════════════════════════════════════════════
# STEP 6 – plot
# ═══════════════════════════════════════════════════════════════════════════════

def plot_correlation(elev, diff, lat_vals,
                     pearson_r, pearson_p,
                     spearman_r, spearman_p,
                     slope, intercept):

    fig, ax = plt.subplots(figsize=(8, 6))

    # ── scatter coloured by |latitude| ────────────────────────────────────────
    sc = ax.scatter(
        elev, diff,
        c=np.abs(lat_vals),
        cmap="plasma_r",
        norm=mcolors.Normalize(vmin=0, vmax=90),
        s=2, alpha=0.35, linewidths=0,
        rasterized=True,
    )

    # ── OLS regression line ───────────────────────────────────────────────────
    x_line = np.array([elev.min(), elev.max()])
    ax.plot(x_line, slope * x_line + intercept,
            color="#1a1a1a", linewidth=1.8, linestyle="--",
            label="OLS fit", zorder=5)

    # ── zero reference ────────────────────────────────────────────────────────
    ax.axhline(0, color="#888888", linewidth=0.9, linestyle=":", zorder=4)

    # ── stats annotation ──────────────────────────────────────────────────────
    def _pstr(p):
        return "< 0.001" if p < 0.001 else f"= {p:.3f}"

    ax.text(
        0.97, 0.97,
        (f"Pearson $r$ = {pearson_r:+.3f}  ($p$ {_pstr(pearson_p)})\n"
         f"Spearman $\\rho$ = {spearman_r:+.3f}  ($p$ {_pstr(spearman_p)})\n"
         f"OLS slope = {slope:.2e} °C / lum. unit"),
        transform=ax.transAxes,
        fontsize=10.5, va="top", ha="right", linespacing=1.6,
        bbox=dict(boxstyle="round,pad=0.4", facecolor="white",
                  edgecolor="#cccccc", alpha=0.90),
    )

    # ── panel label ───────────────────────────────────────────────────────────
    ax.text(-0.08, 1.05, "a",
            transform=ax.transAxes,
            fontsize=14, fontweight="bold", va="top", ha="left")

    # ── formatting ────────────────────────────────────────────────────────────
    ax.set_xlabel("Elevation proxy — NaturalEarth luminance (0 = low, 255 = high)",
                  fontsize=12, labelpad=4)
    ax.set_ylabel("Mean UTCI difference — NN − Polynomial (°C)",
                  fontsize=12, labelpad=4)
    ax.set_title(
        "Elevation vs Mean UTCI Approximation Difference (2024, land only)",
        pad=6,
    )
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="both", length=4, width=0.8)

    # ── colourbar ─────────────────────────────────────────────────────────────
    cbar = fig.colorbar(sc, ax=ax, pad=0.02, shrink=0.85)
    cbar.set_label("|Latitude| (°)", fontsize=11, labelpad=4)
    cbar.ax.tick_params(labelsize=10)

    plt.tight_layout(pad=1.2)

    # ── save ──────────────────────────────────────────────────────────────────
    for ext in [".png", ".pdf", ".eps"]:
        out = OUTPUT_BASE + ext
        fig.savefig(out, dpi=600, bbox_inches="tight",
                    pad_inches=0.01, facecolor="white")
        print(f"  ✓  Saved → {out}")

    plt.show()


# ═══════════════════════════════════════════════════════════════════════════════
# ENTRY POINT
# ═══════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    # 1 – UTCI data
    diff_grid, lats, lons = load_utci()

    # 2 + 3 – elevation proxy from NaturalEarth raster
    elev_grid = build_elevation_grid(lats, lons)

    # 4 – land mask
    land_mask, lat2d = build_land_mask(lats, lons)

    # flatten + apply mask
    elev_flat = elev_grid.ravel()
    diff_flat = diff_grid.ravel()
    lat_flat  = lat2d.ravel()
    mask_flat = (
        land_mask.ravel()        &
        np.isfinite(elev_flat)   &
        np.isfinite(diff_flat)
    )
    elev_land = elev_flat[mask_flat]
    diff_land = diff_flat[mask_flat]
    lat_land  = lat_flat[mask_flat]
    print(f"Final land pixels: {mask_flat.sum():,}")

    # 5 – stats (printed to console)
    pr, pp, sr, sp, slope, intercept = compute_stats(elev_land, diff_land)

    # 6 – plot + save
    plot_correlation(elev_land, diff_land, lat_land,
                     pr, pp, sr, sp, slope, intercept)

In [ ]:
"""
utci_elevation_correlation.py
==============================
Correlation between land-surface elevation and the annual mean UTCI
approximation difference (NN − Polynomial).

Elevation source — cartopy NaturalEarth raster (zero extra installs)
---------------------------------------------------------------------
Cartopy automatically downloads and caches the NaturalEarth
Cross-Blended Hypsometric Tints + Shaded Relief GeoTIFF (~30 MB) the
first time this script runs.  No account, no API key, no pip install
beyond what the six-panel figure already requires.

The raster is a visual product (not a true DEM) so pixel luminance is
used as an elevation proxy: bright = high terrain, dark = low/ocean.
At ERA5 resolution (~0.25°) this gives a reliable elevation signal for
correlation analysis.

Steps
-----
1. Load mean UTCI difference      (utci_diff_2024_mean.nc)
2. Download/cache NE raster via cartopy, read with Pillow
3. Interpolate raster luminance to UTCI grid
4. Apply land mask (cartopy 110 m shapefile — already used in six-panel fig)
5. Pearson r, Spearman ρ, OLS regression → print to console
6. Scatter plot coloured by |latitude|, OLS line, stats annotated
7. Save PNG + PDF + EPS at 600 dpi

Only edit the CONFIG block, then run:
    python utci_elevation_correlation.py
"""

import os
import io
import zipfile
import urllib.request
import numpy as np
import xarray as xr
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy import stats

import cartopy
from cartopy.io import shapereader
from shapely.geometry import Point
from shapely.ops import unary_union
from shapely.prepared import prep

# ═══════════════════════════════════════════════════════════════════════════════
# CONFIG  ── only edit this block ───────────────────────────────────────────────
# ═══════════════════════════════════════════════════════════════════════════════

MEAN_STD_DIR = (
    r"C:\Users\nerc-user\OneDrive - Nexus365"
    r"\UTCI_NN\ERA5_exploration\data\utci_diff_mean_var"
)
MEAN_NC     = os.path.join(MEAN_STD_DIR, "utci_diff_2024_mean.nc")
OUTPUT_BASE = "utci_elevation_correlation"   # .png / .pdf / .eps appended

# ═══════════════════════════════════════════════════════════════════════════════
# STYLE  ── consistent with companion figures ───────────────────────────────────
# ═══════════════════════════════════════════════════════════════════════════════

mpl.rcParams.update({
    "figure.dpi":        300,
    "savefig.dpi":       600,
    "font.family":       "serif",
    "font.serif":        ["Times New Roman", "Times", "DejaVu Serif"],
    "font.size":         12,
    "axes.titlesize":    13,
    "axes.labelsize":    12,
    "axes.linewidth":    1.0,
    "xtick.labelsize":   11,
    "ytick.labelsize":   11,
    "legend.fontsize":   11,
    "legend.frameon":    False,
    "lines.linewidth":   1.5,
    "axes.grid":         False,
    "pdf.fonttype":      42,
    "ps.fonttype":       42,
    "xtick.major.width": 1.0,
    "ytick.major.width": 1.0,
})

# ═══════════════════════════════════════════════════════════════════════════════
# HELPERS
# ═══════════════════════════════════════════════════════════════════════════════

def _get_coord(ds, options):
    for name in options:
        if name in ds.coords:
            return name
    raise KeyError(f"None of {options} found in {list(ds.coords)}")


# ═══════════════════════════════════════════════════════════════════════════════
# STEP 1 – load UTCI mean difference
# ═══════════════════════════════════════════════════════════════════════════════

def load_utci():
    print("Loading UTCI mean difference …")
    ds       = xr.open_dataset(MEAN_NC)
    lat_name = _get_coord(ds, ["lat", "latitude", "y"])
    lon_name = _get_coord(ds, ["lon", "longitude", "x"])
    diff     = ds["approx_diff_mean"].squeeze().values
    lats     = ds[lat_name].values
    lons     = ds[lon_name].values
    ds.close()
    print(f"  Grid: {len(lats)} lats × {len(lons)} lons")
    return diff, lats, lons


# ═══════════════════════════════════════════════════════════════════════════════
# STEP 2 – get NaturalEarth raster path (download once via cartopy cache)
# ═══════════════════════════════════════════════════════════════════════════════

def _get_ne_raster_path():
    """
    Return local path to the NaturalEarth shaded-relief GeoTIFF,
    downloading and caching it if not already present.

    Tries three URLs in order:
      1. naturalearth.s3.amazonaws.com  (primary)
      2. naciscdn.org mirror             (fallback)
      3. GitHub Natural Earth repo       (fallback)
    If all fail, prints manual download instructions and raises.
    """
    cache_dir = os.path.join(cartopy.config["data_dir"], "ne_raster")
    os.makedirs(cache_dir, exist_ok=True)

    # ── return immediately if already cached ──────────────────────────────────
    for fname in ["NE1_HR_LC_SR_W_DR.tif",
                  "NE1_HR_LC_SR_W_DR.tiff",
                  "NE1_HR_LC_SR_W_DR.jpg"]:
        candidate = os.path.join(cache_dir, fname)
        if os.path.exists(candidate):
            print(f"  Using cached raster: {candidate}")
            return candidate

    # ── try multiple known URLs ───────────────────────────────────────────────
    # Natural Earth changed their S3 layout; the zip is now versioned.
    # We try several known-good locations in order.
    candidate_urls = [
        # Current NaturalEarth S3 path (as of 2025)
        "https://naturalearth.s3.amazonaws.com/raster/NE1_HR_LC_SR_W_DR.zip",
        # NACIS CDN mirror (often more stable)
        "https://naciscdn.org/naturalearth/packages/NE1_HR_LC_SR_W_DR.zip",
        # Smaller 1:50m version as a lighter fallback (~8 MB)
        "https://naturalearth.s3.amazonaws.com/50m_raster/NE1_50M_SR_W.zip",
        "https://naciscdn.org/naturalearth/packages/NE1_50M_SR_W.zip",
    ]

    last_err = None
    for url in candidate_urls:
        print(f"  Trying: {url}")
        try:
            req = urllib.request.Request(
                url,
                headers={"User-Agent": "Mozilla/5.0 (utci-elevation-script)"}
            )
            with urllib.request.urlopen(req, timeout=60) as resp:
                raw = resp.read()
            print(f"  Downloaded {len(raw) / 1e6:.1f} MB — extracting …")
            with zipfile.ZipFile(io.BytesIO(raw)) as zf:
                for member in zf.namelist():
                    if member.lower().endswith((".tif", ".tiff", ".jpg", ".png")):
                        zf.extract(member, cache_dir)
                        # flatten any sub-directory in the zip
                        extracted = os.path.join(cache_dir, member)
                        flat = os.path.join(cache_dir, os.path.basename(member))
                        if extracted != flat:
                            os.rename(extracted, flat)
                        print(f"  Saved → {flat}")
                        return flat
        except Exception as e:
            print(f"  ✗ Failed ({type(e).__name__}: {e})")
            last_err = e
            continue

    # ── all URLs failed: give clear manual instructions ───────────────────────
    raise FileNotFoundError(
        "All automatic download attempts failed.\n\n"
        "MANUAL FIX (takes 2 minutes):\n"
        "  1. Open https://www.naturalearthdata.com/downloads/10m-raster-data/"
        "10m-natural-earth-1/\n"
        "  2. Download \'Natural Earth I with Shaded Relief, Water, and Drainages\'\n"
        "  3. Extract the .tif file\n"
        f"  4. Copy it to: {cache_dir}\\NE1_HR_LC_SR_W_DR.tif\n"
        "  5. Re-run this script\n\n"
        f"Last error: {last_err}"
    )


# ═══════════════════════════════════════════════════════════════════════════════
# STEP 3 – build elevation proxy grid
# ═══════════════════════════════════════════════════════════════════════════════

def build_elevation_grid(lats, lons):
    """
    Read the NaturalEarth raster and interpolate luminance (elevation proxy)
    onto the UTCI lat/lon grid.

    Requires Pillow — installed alongside matplotlib in most scientific
    Python environments.  If missing: pip install Pillow
    """
    try:
        from PIL import Image
    except ImportError:
        raise ImportError(
            "Pillow is required to read the NaturalEarth raster.\n"
            "Install it with:  pip install Pillow"
        )

    img_path = _get_ne_raster_path()
    print("  Reading raster and computing luminance …")
    img     = Image.open(img_path).convert("RGB")
    arr     = np.array(img, dtype=np.float32)          # (H, W, 3)
    # ITU-R BT.601 luminance: bright pixels = high terrain
    lum     = 0.299 * arr[:, :, 0] + 0.587 * arr[:, :, 1] + 0.114 * arr[:, :, 2]

    H, W    = lum.shape
    # NaturalEarth global raster: lon −180…180 left→right, lat +90…−90 top→bottom
    img_lons = np.linspace(-180.0, 180.0, W, endpoint=False) + 180.0 / W
    img_lats = np.linspace( 90.0, -90.0, H, endpoint=False) -  90.0 / H

    print("  Interpolating raster to UTCI grid …")
    da_lum = xr.DataArray(
        lum,
        dims=("ilat", "ilon"),
        coords={"ilat": img_lats, "ilon": img_lons},
    )
    # Linear interpolation — fast with xarray on a regular grid
    elev_grid = da_lum.interp(ilat=lats, ilon=lons, method="linear").values
    print(f"  Luminance range on UTCI grid: "
          f"{np.nanmin(elev_grid):.1f} – {np.nanmax(elev_grid):.1f}")
    return elev_grid


# ═══════════════════════════════════════════════════════════════════════════════
# STEP 4 – land mask (cartopy 110 m shapefile, same as six-panel figure)
# ═══════════════════════════════════════════════════════════════════════════════

def build_land_mask(lats, lons):
    print("Building land mask (cartopy 110 m shapefile) …")
    shp   = shapereader.natural_earth("110m", "physical", "land")
    geoms = [g.geometry for g in shapereader.Reader(shp).records()]
    land  = prep(unary_union(geoms))

    lon2d, lat2d = np.meshgrid(lons, lats)
    flat  = lon2d.size
    mask  = np.empty(flat, dtype=bool)
    chunk = 500_000
    for i in range(0, flat, chunk):
        pts = [Point(xy) for xy in zip(lon2d.ravel()[i:i+chunk],
                                       lat2d.ravel()[i:i+chunk])]
        mask[i:i+chunk] = [land.contains(p) for p in pts]

    print(f"  Land pixels: {mask.sum():,} / {flat:,}")
    return mask.reshape(lat2d.shape), lat2d


# ═══════════════════════════════════════════════════════════════════════════════
# STEP 5 – statistics
# ═══════════════════════════════════════════════════════════════════════════════

def compute_stats(elev, diff):
    pearson_r,  pearson_p  = stats.pearsonr(elev, diff)
    spearman_r, spearman_p = stats.spearmanr(elev, diff)
    slope, intercept, *_   = stats.linregress(elev, diff)

    print("\n── Correlation statistics ──────────────────────────────────────────────")
    print(f"  N (land pixels)  : {len(elev):,}")
    print(f"  Pearson  r       : {pearson_r:+.4f}   (p = {pearson_p:.2e})")
    print(f"  Spearman ρ       : {spearman_r:+.4f}   (p = {spearman_p:.2e})")
    print(f"  OLS slope        : {slope:+.4e}  °C per luminance unit")
    print(f"  OLS intercept    : {intercept:+.4f} °C")
    print(f"  Note: elevation proxy = NaturalEarth luminance")
    print(f"        (0 = dark / low terrain, 255 = bright / high terrain)")
    print("────────────────────────────────────────────────────────────────────────\n")

    return pearson_r, pearson_p, spearman_r, spearman_p, slope, intercept


# ═══════════════════════════════════════════════════════════════════════════════
# STEP 6 – plot
# ═══════════════════════════════════════════════════════════════════════════════

def plot_correlation(elev, diff, lat_vals,
                     pearson_r, pearson_p,
                     spearman_r, spearman_p,
                     slope, intercept):

    fig, ax = plt.subplots(figsize=(8, 6))

    # ── scatter coloured by |latitude| ────────────────────────────────────────
    sc = ax.scatter(
        elev, diff,
        c=np.abs(lat_vals),
        cmap="plasma_r",
        norm=mcolors.Normalize(vmin=0, vmax=90),
        s=2, alpha=0.35, linewidths=0,
        rasterized=True,
    )

    # ── OLS regression line ───────────────────────────────────────────────────
    x_line = np.array([elev.min(), elev.max()])
    ax.plot(x_line, slope * x_line + intercept,
            color="#1a1a1a", linewidth=1.8, linestyle="--",
            label="OLS fit", zorder=5)

    # ── zero reference ────────────────────────────────────────────────────────
    ax.axhline(0, color="#888888", linewidth=0.9, linestyle=":", zorder=4)

    # ── stats annotation ──────────────────────────────────────────────────────
    def _pstr(p):
        return "< 0.001" if p < 0.001 else f"= {p:.3f}"

    ax.text(
        0.97, 0.97,
        (f"Pearson $r$ = {pearson_r:+.3f}  ($p$ {_pstr(pearson_p)})\n"
         f"Spearman $\\rho$ = {spearman_r:+.3f}  ($p$ {_pstr(spearman_p)})\n"
         f"OLS slope = {slope:.2e} °C / lum. unit"),
        transform=ax.transAxes,
        fontsize=10.5, va="top", ha="right", linespacing=1.6,
        bbox=dict(boxstyle="round,pad=0.4", facecolor="white",
                  edgecolor="#cccccc", alpha=0.90),
    )

    # ── panel label ───────────────────────────────────────────────────────────
    ax.text(-0.08, 1.05, "a",
            transform=ax.transAxes,
            fontsize=14, fontweight="bold", va="top", ha="left")

    # ── formatting ────────────────────────────────────────────────────────────
    ax.set_xlabel("Elevation proxy — NaturalEarth luminance (0 = low, 255 = high)",
                  fontsize=12, labelpad=4)
    ax.set_ylabel("Mean UTCI difference — NN − Polynomial (°C)",
                  fontsize=12, labelpad=4)
    ax.set_title(
        "Elevation vs Mean UTCI Approximation Difference (2024, land only)",
        pad=6,
    )
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="both", length=4, width=0.8)

    # ── colourbar ─────────────────────────────────────────────────────────────
    cbar = fig.colorbar(sc, ax=ax, pad=0.02, shrink=0.85)
    cbar.set_label("|Latitude| (°)", fontsize=11, labelpad=4)
    cbar.ax.tick_params(labelsize=10)

    plt.tight_layout(pad=1.2)

    # ── save ──────────────────────────────────────────────────────────────────
    for ext in [".png", ".pdf", ".eps"]:
        out = OUTPUT_BASE + ext
        fig.savefig(out, dpi=600, bbox_inches="tight",
                    pad_inches=0.01, facecolor="white")
        print(f"  ✓  Saved → {out}")

    plt.show()


# ═══════════════════════════════════════════════════════════════════════════════
# ENTRY POINT
# ═══════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    # 1 – UTCI data
    diff_grid, lats, lons = load_utci()

    # 2 + 3 – elevation proxy from NaturalEarth raster
    elev_grid = build_elevation_grid(lats, lons)

    # 4 – land mask
    land_mask, lat2d = build_land_mask(lats, lons)

    # flatten + apply mask
    elev_flat = elev_grid.ravel()
    diff_flat = diff_grid.ravel()
    lat_flat  = lat2d.ravel()
    mask_flat = (
        land_mask.ravel()        &
        np.isfinite(elev_flat)   &
        np.isfinite(diff_flat)
    )
    elev_land = elev_flat[mask_flat]
    diff_land = diff_flat[mask_flat]
    lat_land  = lat_flat[mask_flat]
    print(f"Final land pixels: {mask_flat.sum():,}")

    # 5 – stats (printed to console)
    pr, pp, sr, sp, slope, intercept = compute_stats(elev_land, diff_land)

    # 6 – plot + save
    plot_correlation(elev_land, diff_land, lat_land,
                     pr, pp, sr, sp, slope, intercept)